# Newfoundland recurvature climatology

This notebook reproduces the 1950–2023 track-climatology analysis and,
optionally, the satellite-era 500-hPa composite.

Principal workflow features:

- an explicit tropical-origin sample requiring an IBTrACS `TS` nature code at
  least once;
- regular 6-hour track interpolation without bridging source gaps longer than
  12 hours;
- a fixed-duration, physically constrained recurvature detector using 24-hour
  pre- and post-turn velocity windows;
- distance from the continuous post-turn track to the Newfoundland island
  polygon;
- separate models for event counts, conditional pathway probabilities,
  recurvature location, and the untruncated proximity distribution;
- full-period and satellite-era detector, temporal-window,
  velocity-threshold, and regional-distance sensitivity analyses;
- diagnostics for potential end-of-track distance censoring and alternative
  map projection;
- a 500-hPa anomaly composite with both fixed geographic and
  storm-relative coordinates.

The notebook downloads public IBTrACS and NCEP/NCAR Reanalysis data. The track
analysis is substantially faster than the optional composite, which must cache
one field for each satellite-era event.

## 1. Install and configure

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "cartopy": "cartopy",
    "netCDF4": "netCDF4",
    "pyproj": "pyproj",
    "shapely": "shapely",
    "statsmodels": "statsmodels",
    "xarray": "xarray",
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )
print("Dependencies ready")

In [ ]:
import os
from pathlib import Path

if "COLAB_RELEASE_TAG" in os.environ:
    PROJECT_ROOT = Path("/content/newfoundland_tc_recurvature")
else:
    PROJECT_ROOT = Path(
        os.environ.get("NL_RECURV_WORKDIR", Path.cwd() / "nl_recurvature")
    ).resolve()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["NL_RECURV_WORKDIR"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

## 2. Materialize the audited analysis source

The three source modules are embedded in this notebook so that the notebook is
portable and does not depend on a separate repository checkout.

In [ ]:
from pathlib import Path

source = '"""Core analysis for the Newfoundland tropical-cyclone recurvature study.\n\nThe publication workflow has four defining features:\n\n1. Every track is placed on a regular 6-hour UTC grid.\n2. Recurvature windows are defined in hours and require a west-to-east\n   transition with continued poleward motion.\n3. Proximity is measured from the post-turn track polyline to the\n   Newfoundland island polygon.\n4. Frequency, pathway rate, recurvature location, and proximity are each\n   quantified with effect estimates and uncertainty.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport urllib.request\nimport warnings\nfrom dataclasses import asdict, dataclass\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cartopy\nimport cartopy.crs as ccrs\nimport cartopy.feature as cfeature\nfrom cartopy.io import shapereader\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom pyproj import Transformer\nfrom scipy import stats\nfrom scipy.spatial.distance import cdist, pdist\nfrom shapely.geometry import LineString, MultiPolygon, Point, Polygon\nfrom shapely.ops import nearest_points, transform as shapely_transform\nimport statsmodels.api as sm\nfrom statsmodels.stats.diagnostic import acorr_ljungbox\nimport xarray as xr\n\n\nR_EARTH_KM = 6371.0\nYEAR_MIN = 1950\nYEAR_MAX = 2023\nSATELLITE_START = 1979\n\nIBTRACS_URL = (\n    "https://www.ncei.noaa.gov/data/"\n    "international-best-track-archive-for-climate-stewardship-ibtracs/"\n    "v04r00/access/netcdf/IBTrACS.ALL.v04r00.nc"\n)\n\n\n@dataclass(frozen=True)\nclass DetectorConfig:\n    grid_hours: int = 6\n    window_hours: int = 24\n    latitude_gate_deg_n: float = 25.0\n    pre_east_max_kmh: float = 0.0\n    post_east_min_kmh: float = 5.0\n    post_north_min_kmh: float = 0.0\n    min_eastward_acceleration_kmh: float = 5.0\n    maximum_source_gap_hours: float = 12.0\n\n    @property\n    def window_steps(self) -> int:\n        steps = self.window_hours / self.grid_hours\n        if not float(steps).is_integer():\n            raise ValueError("window_hours must be divisible by grid_hours")\n        return int(steps)\n\n\n@dataclass(frozen=True)\nclass RegionConfig:\n    proximity_threshold_km: float = 600.0\n    projection_epsg: int = 3347\n\n\nBASELINE_DETECTOR = DetectorConfig()\nBASELINE_REGION = RegionConfig()\n\nLEGACY_NL_LAT_MIN = 44.0\nLEGACY_NL_LAT_MAX = 54.5\nLEGACY_NL_LON_MIN = -61.5\nLEGACY_NL_LON_MAX = -50.0\nLEGACY_PROXY_POINTS = np.array(\n    [\n        [47.56, -52.71],\n        [48.95, -57.95],\n        [48.70, -53.11],\n        [49.18, -55.74],\n        [51.45, -56.00],\n    ],\n    dtype=float,\n)\n\n\ndef decode_text(value) -> str:\n    if isinstance(value, (bytes, np.bytes_)):\n        return value.decode("utf-8", errors="ignore").strip(" \\x00")\n    if hasattr(value, "tobytes") and getattr(value, "dtype", None) is not None:\n        if value.dtype.kind == "S":\n            return value.tobytes().decode("utf-8", errors="ignore").strip(" \\x00")\n    return str(value).strip()\n\n\ndef wrap_longitude(lon):\n    lon = np.asarray(lon, dtype=float)\n    return ((lon + 180.0) % 360.0) - 180.0\n\n\ndef download_if_needed(url: str, destination: Path) -> Path:\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if not destination.exists():\n        print(f"Downloading {url}")\n        urllib.request.urlretrieve(url, destination)\n    return destination\n\n\ndef load_ibtracs(path: Path) -> xr.Dataset:\n    return xr.open_dataset(path, decode_times=True)\n\n\ndef north_atlantic_indices(\n    ds: xr.Dataset,\n    year_min: int = YEAR_MIN,\n    year_max: int = YEAR_MAX,\n) -> np.ndarray:\n    season = ds["season"].values.astype(int)\n    basin = ds["basin"].values\n    has_na = (basin == b"NA").any(axis=1)\n    return np.where((season >= year_min) & (season <= year_max) & has_na)[0]\n\n\ndef tropical_origin_indices(\n    ds: xr.Dataset,\n    storm_indices: Iterable[int],\n) -> np.ndarray:\n    """Retain storms coded tropical (IBTrACS nature code TS) at least once."""\n\n    if "nature" not in ds.variables:\n        raise KeyError("IBTrACS nature is required to define tropical origin")\n    retained = []\n    for k in storm_indices:\n        k = int(k)\n        if any(decode_text(value) == "TS" for value in ds["nature"].values[k]):\n            retained.append(k)\n    return np.asarray(retained, dtype=int)\n\n\ndef _round_times_to_minute(values: np.ndarray) -> pd.DatetimeIndex:\n    return pd.DatetimeIndex(pd.to_datetime(values)).round("min")\n\n\ndef _contiguous_source_segments(\n    frame: pd.DataFrame,\n    maximum_gap_hours: float,\n) -> list[pd.DataFrame]:\n    if frame.empty:\n        return []\n    gaps = frame["time"].diff().dt.total_seconds().div(3600.0)\n    segment_id = (gaps > maximum_gap_hours).cumsum()\n    return [g.copy() for _, g in frame.groupby(segment_id, sort=True)]\n\n\ndef resample_track_six_hourly(\n    times: np.ndarray,\n    lats: np.ndarray,\n    lons: np.ndarray,\n    natures: np.ndarray | None = None,\n    statuses: np.ndarray | None = None,\n    grid_hours: int = 6,\n    maximum_source_gap_hours: float = 12.0,\n) -> pd.DataFrame:\n    """Interpolate a track to a regular UTC grid without bridging long gaps."""\n\n    valid = np.isfinite(lats) & np.isfinite(lons) & ~pd.isna(times)\n    if valid.sum() < 2:\n        return pd.DataFrame()\n\n    frame = pd.DataFrame(\n        {\n            "time": _round_times_to_minute(times[valid]),\n            "lat": np.asarray(lats, dtype=float)[valid],\n            "lon": wrap_longitude(np.asarray(lons, dtype=float)[valid]),\n        }\n    )\n    if natures is not None:\n        frame["nature"] = [decode_text(v) for v in np.asarray(natures)[valid]]\n    if statuses is not None:\n        frame["status"] = [decode_text(v) for v in np.asarray(statuses)[valid]]\n\n    frame = (\n        frame.sort_values("time")\n        .drop_duplicates("time", keep="first")\n        .reset_index(drop=True)\n    )\n    if len(frame) < 2:\n        return pd.DataFrame()\n\n    pieces = []\n    for source in _contiguous_source_segments(frame, maximum_source_gap_hours):\n        if len(source) < 2:\n            continue\n        start = source["time"].iloc[0].ceil(f"{grid_hours}h")\n        end = source["time"].iloc[-1].floor(f"{grid_hours}h")\n        if start > end:\n            continue\n        grid = pd.date_range(start, end, freq=f"{grid_hours}h")\n        if len(grid) < 2:\n            continue\n\n        source_seconds = source["time"].astype("int64").to_numpy(dtype=float) / 1e9\n        grid_seconds = grid.astype("int64").to_numpy(dtype=float) / 1e9\n        unwrapped_lon = np.degrees(\n            np.unwrap(np.radians(source["lon"].to_numpy(dtype=float)))\n        )\n\n        out = pd.DataFrame(\n            {\n                "time": grid,\n                "lat": np.interp(\n                    grid_seconds, source_seconds, source["lat"].to_numpy(dtype=float)\n                ),\n                "lon_unwrapped": np.interp(\n                    grid_seconds, source_seconds, unwrapped_lon\n                ),\n            }\n        )\n        out["lon"] = wrap_longitude(out["lon_unwrapped"])\n\n        nearest = np.abs(\n            source_seconds[:, None] - grid_seconds[None, :]\n        ).argmin(axis=0)\n        if "nature" in source:\n            out["nature"] = source["nature"].to_numpy()[nearest]\n        if "status" in source:\n            out["status"] = source["status"].to_numpy()[nearest]\n        pieces.append(out)\n\n    if not pieces:\n        return pd.DataFrame()\n\n    out = (\n        pd.concat(pieces, ignore_index=True)\n        .sort_values("time")\n        .drop_duplicates("time", keep="first")\n        .reset_index(drop=True)\n    )\n    dt = out["time"].diff().dt.total_seconds().div(3600.0)\n    out["segment"] = (dt > grid_hours * 1.01).cumsum()\n    return out\n\n\ndef displacement_components_km(track: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:\n    lat1 = np.radians(track["lat"].to_numpy(dtype=float)[:-1])\n    lat2 = np.radians(track["lat"].to_numpy(dtype=float)[1:])\n    lon_unwrapped = np.radians(track["lon_unwrapped"].to_numpy(dtype=float))\n    dlon = np.diff(lon_unwrapped)\n    dlat = lat2 - lat1\n    mean_lat = 0.5 * (lat1 + lat2)\n    dx = R_EARTH_KM * np.cos(mean_lat) * dlon\n    dy = R_EARTH_KM * dlat\n    return dx, dy\n\n\ndef haversine_km(lat1, lon1, lat2, lon2):\n    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])\n    dlat = lat2 - lat1\n    dlon = lon2 - lon1\n    value = (\n        np.sin(dlat / 2.0) ** 2\n        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2\n    )\n    return 2.0 * R_EARTH_KM * np.arcsin(np.sqrt(value))\n\n\ndef detect_legacy_increment_recurvature(\n    lat: np.ndarray,\n    lon: np.ndarray,\n    latitude_gate_deg_n: float = 25.0,\n    window_steps: int = 4,\n    post_zonal_km_per_increment: float = 30.0,\n    pre_zonal_max_km_per_increment: float = 30.0,\n) -> dict | None:\n    """Recovered increment-based detector retained only for comparison."""\n\n    if len(lat) < 2 * window_steps + 3:\n        return None\n    lon_unwrapped = np.degrees(np.unwrap(np.radians(lon)))\n    dlon = np.radians(np.diff(lon_unwrapped))\n    mean_lat = 0.5 * (lat[:-1] + lat[1:])\n    zonal_km = R_EARTH_KM * np.cos(np.radians(mean_lat)) * dlon\n    for j in range(window_steps, len(zonal_km) - window_steps):\n        i_star = j + 1\n        if lat[i_star] < latitude_gate_deg_n:\n            continue\n        pre = float(np.mean(zonal_km[j - window_steps : j]))\n        post = float(np.mean(zonal_km[j : j + window_steps]))\n        if (\n            pre <= pre_zonal_max_km_per_increment\n            and post > post_zonal_km_per_increment\n        ):\n            return {\n                "recurv_index": int(i_star),\n                "recurv_lat": float(lat[i_star]),\n                "recurv_lon": float(lon[i_star]),\n                "legacy_pre_zonal_km_per_increment": pre,\n                "legacy_post_zonal_km_per_increment": post,\n            }\n    return None\n\n\ndef classify_legacy_method(\n    ds: xr.Dataset,\n    storm_indices: Iterable[int],\n    distance_threshold_km: float = 600.0,\n    minimum_track_points: int = 25,\n) -> pd.DataFrame:\n    """Apply the diagnostic native-cadence workflow comparator."""\n\n    rows = []\n    for k in storm_indices:\n        k = int(k)\n        lat_raw = ds["lat"].values[k].astype(float)\n        lon_raw = wrap_longitude(ds["lon"].values[k].astype(float))\n        time_raw = ds["time"].values[k]\n        valid = np.isfinite(lat_raw) & np.isfinite(lon_raw)\n        lat = lat_raw[valid]\n        lon = lon_raw[valid]\n        original_indices = np.flatnonzero(valid)\n        if len(lat) < minimum_track_points:\n            continue\n        result = detect_legacy_increment_recurvature(lat, lon)\n        if result is None:\n            continue\n        i_star = result["recurv_index"]\n        post_lat = lat[i_star:]\n        post_lon = lon[i_star:]\n        box_hit = bool(\n            np.any(\n                (post_lat >= LEGACY_NL_LAT_MIN)\n                & (post_lat <= LEGACY_NL_LAT_MAX)\n                & (post_lon >= LEGACY_NL_LON_MIN)\n                & (post_lon <= LEGACY_NL_LON_MAX)\n            )\n        )\n        min_distance = min(\n            float(np.nanmin(haversine_km(post_lat, post_lon, point[0], point[1])))\n            for point in LEGACY_PROXY_POINTS\n        )\n        relevant = box_hit or min_distance <= distance_threshold_km\n        if not relevant:\n            continue\n        original_index = int(original_indices[i_star])\n        rows.append(\n            {\n                "dataset_index": k,\n                "sid": decode_text(ds["sid"].values[k]),\n                "season": int(ds["season"].values[k]),\n                "name": decode_text(ds["name"].values[k]),\n                "recurv_time": pd.Timestamp(time_raw[original_index]).round("min"),\n                "box_hit": box_hit,\n                "minimum_proxy_distance_km": min_distance,\n                **result,\n            }\n        )\n    return pd.DataFrame(rows).sort_values(["season", "sid"]).reset_index(drop=True)\n\n\ndef detect_recurvature_velocity(\n    track: pd.DataFrame,\n    config: DetectorConfig = BASELINE_DETECTOR,\n) -> dict | None:\n    """Detect the first sustained west-to-east, poleward trajectory turn."""\n\n    win = config.window_steps\n    if len(track) < 2 * win + 1:\n        return None\n\n    dx, dy = displacement_components_km(track)\n    dt_hours = float(config.grid_hours)\n    u = dx / dt_hours\n    v = dy / dt_hours\n    segments = track["segment"].to_numpy(dtype=int)\n\n    for boundary in range(win, len(track) - win):\n        # The candidate position is the first point in the post-turn window.\n        if track["lat"].iloc[boundary] < config.latitude_gate_deg_n:\n            continue\n        if segments[boundary - win] != segments[boundary + win]:\n            continue\n\n        pre_u = float(np.mean(u[boundary - win : boundary]))\n        post_u = float(np.mean(u[boundary : boundary + win]))\n        post_v = float(np.mean(v[boundary : boundary + win]))\n        delta_u = post_u - pre_u\n\n        if (\n            pre_u <= config.pre_east_max_kmh\n            and post_u >= config.post_east_min_kmh\n            and post_v > config.post_north_min_kmh\n            and delta_u >= config.min_eastward_acceleration_kmh\n        ):\n            row = track.iloc[boundary]\n            return {\n                "recurv_index": int(boundary),\n                "recurv_time": pd.Timestamp(row["time"]),\n                "recurv_lat": float(row["lat"]),\n                "recurv_lon": float(row["lon"]),\n                "pre_u_kmh": pre_u,\n                "post_u_kmh": post_u,\n                "post_v_kmh": post_v,\n                "delta_u_kmh": delta_u,\n                "nature_at_recurvature": str(row.get("nature", "")),\n                "status_at_recurvature": str(row.get("status", "")),\n            }\n    return None\n\n\ndef detect_recurvature_heading(\n    track: pd.DataFrame,\n    config: DetectorConfig = BASELINE_DETECTOR,\n    minimum_heading_change_deg: float = 30.0,\n) -> dict | None:\n    """Alternative detector based on local-displacement headings."""\n\n    win = config.window_steps\n    if len(track) < 2 * win + 1:\n        return None\n    dx, dy = displacement_components_km(track)\n    segments = track["segment"].to_numpy(dtype=int)\n    angles = np.degrees(np.arctan2(dx, dy)) % 360.0\n    east_component = np.sin(np.radians(angles))\n    north_component = np.cos(np.radians(angles))\n\n    for boundary in range(win, len(track) - win):\n        if track["lat"].iloc[boundary] < config.latitude_gate_deg_n:\n            continue\n        if segments[boundary - win] != segments[boundary + win]:\n            continue\n\n        pre_e = float(np.mean(east_component[boundary - win : boundary]))\n        post_e = float(np.mean(east_component[boundary : boundary + win]))\n        post_n = float(np.mean(north_component[boundary : boundary + win]))\n        pre_angle = math.degrees(\n            math.atan2(\n                float(np.mean(dx[boundary - win : boundary])),\n                float(np.mean(dy[boundary - win : boundary])),\n            )\n        ) % 360.0\n        post_angle = math.degrees(\n            math.atan2(\n                float(np.mean(dx[boundary : boundary + win])),\n                float(np.mean(dy[boundary : boundary + win])),\n            )\n        ) % 360.0\n        signed_change = ((post_angle - pre_angle + 180.0) % 360.0) - 180.0\n\n        if (\n            pre_e <= 0.0\n            and post_e > 0.10\n            and post_n > 0.0\n            and signed_change >= minimum_heading_change_deg\n        ):\n            row = track.iloc[boundary]\n            return {\n                "recurv_index": int(boundary),\n                "recurv_time": pd.Timestamp(row["time"]),\n                "recurv_lat": float(row["lat"]),\n                "recurv_lon": float(row["lon"]),\n                "pre_heading_deg": pre_angle,\n                "post_heading_deg": post_angle,\n                "heading_change_deg": signed_change,\n                "nature_at_recurvature": str(row.get("nature", "")),\n                "status_at_recurvature": str(row.get("status", "")),\n            }\n    return None\n\n\ndef has_detection_opportunity(\n    track: pd.DataFrame,\n    config: DetectorConfig = BASELINE_DETECTOR,\n) -> bool:\n    """Whether a track contains a complete fixed-duration window poleward of the gate."""\n\n    win = config.window_steps\n    if len(track) < 2 * win + 1:\n        return False\n    segments = track["segment"].to_numpy(dtype=int)\n    latitude = track["lat"].to_numpy(dtype=float)\n    for boundary in range(win, len(track) - win):\n        if (\n            latitude[boundary] >= config.latitude_gate_deg_n\n            and segments[boundary - win] == segments[boundary + win]\n        ):\n            return True\n    return False\n\n\ndef load_newfoundland_island_polygon(\n    cartopy_data_dir: Path,\n) -> tuple[Polygon | MultiPolygon, Polygon | MultiPolygon]:\n    """Return Newfoundland island in lon/lat and EPSG:3347 coordinates."""\n\n    cartopy_data_dir.mkdir(parents=True, exist_ok=True)\n    cartopy.config["data_dir"] = str(cartopy_data_dir)\n    shp = shapereader.natural_earth(\n        resolution="10m",\n        category="cultural",\n        name="admin_1_states_provinces",\n    )\n    province = None\n    for record in shapereader.Reader(shp).records():\n        attr = record.attributes\n        if attr.get("admin") == "Canada" and attr.get("postal") == "NL":\n            province = record.geometry\n            break\n    if province is None:\n        raise RuntimeError("Newfoundland and Labrador polygon was not found")\n\n    components = list(province.geoms) if isinstance(province, MultiPolygon) else [province]\n    island_components = [\n        geom\n        for geom in components\n        if geom.bounds[1] < 52.0 and geom.bounds[3] < 52.5\n    ]\n    if not island_components:\n        raise RuntimeError("Newfoundland island component was not identified")\n    island_lonlat = (\n        island_components[0]\n        if len(island_components) == 1\n        else MultiPolygon(island_components)\n    )\n    project = projected_transformer(3347).transform\n    island_projected = shapely_transform(project, island_lonlat)\n    return island_lonlat, island_projected\n\n\n@lru_cache(maxsize=8)\ndef projected_transformer(projection_epsg: int) -> Transformer:\n    return Transformer.from_crs(\n        "EPSG:4326", f"EPSG:{projection_epsg}", always_xy=True\n    )\n\n\ndef post_track_distance_to_island_km(\n    track: pd.DataFrame,\n    recurv_index: int,\n    island_projected,\n    projection_epsg: int = 3347,\n) -> float:\n    return post_track_distance_diagnostics(\n        track,\n        recurv_index,\n        island_projected,\n        projection_epsg=projection_epsg,\n    )["minimum_distance_to_newfoundland_km"]\n\n\ndef post_track_distance_diagnostics(\n    track: pd.DataFrame,\n    recurv_index: int,\n    island_projected,\n    projection_epsg: int = 3347,\n    endpoint_tolerance_km: float = 1.0,\n) -> dict:\n    """Return projected distance and potential end-of-track censoring diagnostics."""\n\n    transformer = projected_transformer(projection_epsg)\n    return post_track_distance_diagnostics_with_transformer(\n        track,\n        recurv_index,\n        island_projected,\n        transformer,\n        endpoint_tolerance_km=endpoint_tolerance_km,\n    )\n\n\ndef post_track_distance_diagnostics_with_transformer(\n    track: pd.DataFrame,\n    recurv_index: int,\n    island_projected,\n    transformer: Transformer,\n    endpoint_tolerance_km: float = 1.0,\n) -> dict:\n    """Distance diagnostics using a supplied lon/lat-to-projected transformer."""\n\n    post = track.iloc[recurv_index:]\n    x, y = transformer.transform(\n        post["lon"].to_numpy(dtype=float),\n        post["lat"].to_numpy(dtype=float),\n    )\n    if len(x) == 1:\n        geometry = Point(float(x[0]), float(y[0]))\n    else:\n        geometry = LineString(np.column_stack([x, y]))\n    distance_km = float(geometry.distance(island_projected) / 1000.0)\n\n    if isinstance(geometry, LineString) and geometry.length > 0:\n        nearest_on_track, _ = nearest_points(geometry, island_projected)\n        along_track_m = float(geometry.project(nearest_on_track))\n        closest_fraction = along_track_m / float(geometry.length)\n        closest_at_endpoint = bool(\n            float(geometry.length) - along_track_m\n            <= endpoint_tolerance_km * 1000.0\n        )\n    else:\n        closest_fraction = 1.0\n        closest_at_endpoint = True\n\n    duration_hours = float(\n        (\n            pd.Timestamp(post["time"].iloc[-1])\n            - pd.Timestamp(post["time"].iloc[0])\n        ).total_seconds()\n        / 3600.0\n    )\n    return {\n        "minimum_distance_to_newfoundland_km": distance_km,\n        "closest_fraction_along_post_track": closest_fraction,\n        "closest_at_track_endpoint": closest_at_endpoint,\n        "post_recurvature_duration_hours": duration_hours,\n    }\n\n\ndef _storm_track_from_dataset(\n    ds: xr.Dataset,\n    k: int,\n    config: DetectorConfig,\n) -> pd.DataFrame:\n    natures = ds["nature"].values[k] if "nature" in ds.variables else None\n    statuses = ds["usa_status"].values[k] if "usa_status" in ds.variables else None\n    return resample_track_six_hourly(\n        ds["time"].values[k],\n        ds["lat"].values[k],\n        ds["lon"].values[k],\n        natures=natures,\n        statuses=statuses,\n        grid_hours=config.grid_hours,\n        maximum_source_gap_hours=config.maximum_source_gap_hours,\n    )\n\n\ndef build_track_cache(\n    ds: xr.Dataset,\n    storm_indices: Iterable[int],\n    config: DetectorConfig = BASELINE_DETECTOR,\n) -> tuple[dict[int, pd.DataFrame], pd.DataFrame]:\n    cache: dict[int, pd.DataFrame] = {}\n    audit_rows = []\n    for k in storm_indices:\n        track = _storm_track_from_dataset(ds, int(k), config)\n        cache[int(k)] = track\n        source_valid = (\n            np.isfinite(ds["lat"].values[k])\n            & np.isfinite(ds["lon"].values[k])\n            & ~pd.isna(ds["time"].values[k])\n        )\n        source_times = _round_times_to_minute(ds["time"].values[k][source_valid])\n        source_dt = (\n            pd.Series(source_times)\n            .diff()\n            .dt.total_seconds()\n            .div(3600.0)\n            .dropna()\n        )\n        eligible = has_detection_opportunity(track, config)\n        audit_rows.append(\n            {\n                "dataset_index": int(k),\n                "source_points": int(source_valid.sum()),\n                "resampled_points": int(len(track)),\n                "source_min_dt_h": float(source_dt.min()) if len(source_dt) else np.nan,\n                "source_median_dt_h": float(source_dt.median()) if len(source_dt) else np.nan,\n                "source_max_dt_h": float(source_dt.max()) if len(source_dt) else np.nan,\n                "eligible": bool(eligible),\n            }\n        )\n    return cache, pd.DataFrame(audit_rows)\n\n\ndef classify_recurving_storms(\n    ds: xr.Dataset,\n    storm_indices: Iterable[int],\n    track_cache: dict[int, pd.DataFrame],\n    island_projected,\n    detector: str = "velocity",\n    detector_config: DetectorConfig = BASELINE_DETECTOR,\n    region_config: RegionConfig = BASELINE_REGION,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    rows = []\n    eligibility = []\n    season_values = ds["season"].values.astype(int)\n    for k in storm_indices:\n        k = int(k)\n        track = track_cache[k]\n        eligible = has_detection_opportunity(track, detector_config)\n        eligibility.append(\n            {\n                "dataset_index": k,\n                "sid": decode_text(ds["sid"].values[k]),\n                "season": int(season_values[k]),\n                "name": decode_text(ds["name"].values[k]),\n                "eligible": bool(eligible),\n            }\n        )\n        if not eligible:\n            continue\n\n        if detector == "velocity":\n            result = detect_recurvature_velocity(track, detector_config)\n        elif detector == "heading":\n            result = detect_recurvature_heading(track, detector_config)\n        else:\n            raise ValueError("detector must be \'velocity\' or \'heading\'")\n        if result is None:\n            continue\n\n        distance_diagnostics = post_track_distance_diagnostics(\n            track,\n            result["recurv_index"],\n            island_projected,\n            projection_epsg=region_config.projection_epsg,\n        )\n        distance_km = distance_diagnostics[\n            "minimum_distance_to_newfoundland_km"\n        ]\n        row = {\n            "dataset_index": k,\n            "sid": decode_text(ds["sid"].values[k]),\n            "season": int(season_values[k]),\n            "name": decode_text(ds["name"].values[k]),\n            "detector": detector,\n            "newfoundland_relevant": bool(\n                distance_km <= region_config.proximity_threshold_km\n            ),\n            **distance_diagnostics,\n            **result,\n        }\n        rows.append(row)\n\n    recurvers = pd.DataFrame(rows)\n    if len(recurvers):\n        recurvers = recurvers.sort_values(["season", "sid"]).reset_index(drop=True)\n    eligible = pd.DataFrame(eligibility).sort_values(["season", "sid"]).reset_index(drop=True)\n    return recurvers, eligible\n\n\ndef annual_counts(\n    events: pd.DataFrame,\n    year_min: int = YEAR_MIN,\n    year_max: int = YEAR_MAX,\n) -> pd.DataFrame:\n    years = pd.Index(range(year_min, year_max + 1), name="season")\n    values = events["season"].value_counts() if len(events) else pd.Series(dtype=int)\n    return values.reindex(years, fill_value=0).rename("count").reset_index()\n\n\ndef poisson_trend(\n    counts: pd.DataFrame,\n    count_col: str = "count",\n    offset: np.ndarray | None = None,\n    hac_lags: int = 2,\n) -> tuple[dict, sm.GLM]:\n    year = counts["season"].to_numpy(dtype=float)\n    response = counts[count_col].to_numpy(dtype=float)\n    centered = year - year.mean()\n    design = sm.add_constant(centered)\n    model = sm.GLM(\n        response,\n        design,\n        family=sm.families.Poisson(),\n        offset=offset,\n    )\n    result = model.fit()\n    robust = model.fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})\n    beta = float(result.params[1])\n    se = float(result.bse[1])\n    robust_se = float(robust.bse[1])\n    pearson_dispersion = float(result.pearson_chi2 / result.df_resid)\n    prediction = result.get_prediction(design, offset=offset).summary_frame()\n    residuals = np.asarray(result.resid_pearson, dtype=float)\n    lag1 = float(pd.Series(residuals).autocorr(lag=1))\n    lb = acorr_ljungbox(residuals, lags=[min(5, max(1, len(residuals) // 8))])\n\n    summary = {\n        "n_years": int(len(counts)),\n        "total_events": int(response.sum()),\n        "irr_per_decade": float(np.exp(beta * 10.0)),\n        "ci_low": float(np.exp((beta - 1.96 * se) * 10.0)),\n        "ci_high": float(np.exp((beta + 1.96 * se) * 10.0)),\n        "p_value": float(result.pvalues[1]),\n        "hac_ci_low": float(np.exp((beta - 1.96 * robust_se) * 10.0)),\n        "hac_ci_high": float(np.exp((beta + 1.96 * robust_se) * 10.0)),\n        "hac_p_value": float(robust.pvalues[1]),\n        "pearson_dispersion": pearson_dispersion,\n        "lag1_residual_autocorrelation": lag1,\n        "ljung_box_p_value": float(lb["lb_pvalue"].iloc[0]),\n    }\n    fitted = counts[["season", count_col]].copy()\n    fitted["fitted"] = prediction["mean"].to_numpy()\n    fitted["ci_low"] = prediction["mean_ci_lower"].to_numpy()\n    fitted["ci_high"] = prediction["mean_ci_upper"].to_numpy()\n    return summary, fitted\n\n\ndef binomial_pathway_trend(\n    annual_frame: pd.DataFrame,\n    successes: str,\n    trials: str,\n) -> dict:\n    valid = annual_frame[trials] > 0\n    frame = annual_frame.loc[valid].copy()\n    year = frame["season"].to_numpy(dtype=float)\n    centered = year - year.mean()\n    design = sm.add_constant(centered)\n    proportion = frame[successes].to_numpy(dtype=float) / frame[trials].to_numpy(dtype=float)\n    model = sm.GLM(\n        proportion,\n        design,\n        family=sm.families.Binomial(),\n        freq_weights=frame[trials].to_numpy(dtype=float),\n    )\n    result = model.fit()\n    beta = float(result.params[1])\n    se = float(result.bse[1])\n    return {\n        "n_years": int(len(frame)),\n        "total_successes": int(frame[successes].sum()),\n        "total_trials": int(frame[trials].sum()),\n        "odds_ratio_per_decade": float(np.exp(beta * 10.0)),\n        "ci_low": float(np.exp((beta - 1.96 * se) * 10.0)),\n        "ci_high": float(np.exp((beta + 1.96 * se) * 10.0)),\n        "p_value": float(result.pvalues[1]),\n    }\n\n\ndef robust_linear_trend(\n    frame: pd.DataFrame,\n    response: str,\n    scale_per_decade: float = 10.0,\n) -> dict:\n    clean = frame[["season", response]].dropna()\n    year = clean["season"].to_numpy(dtype=float)\n    centered = year - year.mean()\n    design = sm.add_constant(centered)\n    result = sm.OLS(clean[response].to_numpy(dtype=float), design).fit(cov_type="HC3")\n    slope = float(result.params[1]) * scale_per_decade\n    se = float(result.bse[1]) * scale_per_decade\n    return {\n        "n": int(len(clean)),\n        "slope_per_decade": slope,\n        "ci_low": slope - 1.96 * se,\n        "ci_high": slope + 1.96 * se,\n        "p_value": float(result.pvalues[1]),\n    }\n\n\ndef quantile_distance_trend(frame: pd.DataFrame, response: str) -> dict:\n    clean = frame[["season", response]].dropna()\n    year = clean["season"].to_numpy(dtype=float)\n    centered = year - year.mean()\n    design = sm.add_constant(centered)\n    result = sm.QuantReg(clean[response].to_numpy(dtype=float), design).fit(q=0.5)\n    slope = float(result.params[1]) * 10.0\n    se = float(result.bse[1]) * 10.0\n    return {\n        "n": int(len(clean)),\n        "median_slope_km_per_decade": slope,\n        "ci_low": slope - 1.96 * se,\n        "ci_high": slope + 1.96 * se,\n        "p_value": float(result.pvalues[1]),\n    }\n\n\ndef energy_distance_test(\n    first: np.ndarray,\n    second: np.ndarray,\n    permutations: int = 4999,\n    seed: int = 42,\n) -> dict:\n    """Two-sample multivariate energy-distance permutation test."""\n\n    first = np.asarray(first, dtype=float)\n    second = np.asarray(second, dtype=float)\n    n, m = len(first), len(second)\n    pooled = np.vstack([first, second])\n\n    def statistic(a, b):\n        cross = 2.0 * cdist(a, b).mean()\n        within_a = 0.0 if len(a) < 2 else 2.0 * pdist(a).sum() / (len(a) ** 2)\n        within_b = 0.0 if len(b) < 2 else 2.0 * pdist(b).sum() / (len(b) ** 2)\n        return cross - within_a - within_b\n\n    observed = float(statistic(first, second))\n    rng = np.random.default_rng(seed)\n    exceed = 0\n    for _ in range(permutations):\n        permutation = rng.permutation(n + m)\n        value = statistic(pooled[permutation[:n]], pooled[permutation[n:]])\n        exceed += value >= observed\n    return {\n        "n_first": int(n),\n        "n_second": int(m),\n        "energy_statistic": observed,\n        "permutation_p_value": float((exceed + 1) / (permutations + 1)),\n        "permutations": int(permutations),\n    }\n\n\ndef lonlat_to_cartesian_km(\n    longitude_deg: np.ndarray,\n    latitude_deg: np.ndarray,\n) -> np.ndarray:\n    """Map lon/lat to three-dimensional Earth-centred coordinates in kilometres."""\n\n    longitude = np.radians(np.asarray(longitude_deg, dtype=float))\n    latitude = np.radians(np.asarray(latitude_deg, dtype=float))\n    cos_latitude = np.cos(latitude)\n    return R_EARTH_KM * np.column_stack(\n        [\n            cos_latitude * np.cos(longitude),\n            cos_latitude * np.sin(longitude),\n            np.sin(latitude),\n        ]\n    )\n\n\ndef seasonal_counts(events: pd.DataFrame) -> pd.DataFrame:\n    month = pd.to_datetime(events["recurv_time"]).dt.month\n    counts = month.value_counts().reindex(range(1, 13), fill_value=0).sort_index()\n    frame = pd.DataFrame(\n        {\n            "month": counts.index,\n            "month_name": [\n                pd.Timestamp(2000, value, 1).strftime("%B") for value in counts.index\n            ],\n            "events": counts.to_numpy(dtype=int),\n        }\n    )\n    frame["percent"] = 100.0 * frame["events"] / frame["events"].sum()\n    return frame\n\n\ndef status_summary(events: pd.DataFrame) -> pd.DataFrame:\n    values = (\n        events["nature_at_recurvature"]\n        .replace("", "unknown")\n        .fillna("unknown")\n        .value_counts()\n    )\n    return values.rename_axis("nature_code").rename("events").reset_index()\n\n\ndef annual_analysis_frame(\n    relevant: pd.DataFrame,\n    recurvers: pd.DataFrame,\n    eligible: pd.DataFrame,\n    year_min: int = YEAR_MIN,\n    year_max: int = YEAR_MAX,\n) -> pd.DataFrame:\n    frame = pd.DataFrame({"season": np.arange(year_min, year_max + 1)})\n    for label, source in (\n        ("relevant_events", relevant),\n        ("recurving_storms", recurvers),\n        ("eligible_storms", eligible[eligible["eligible"]]),\n    ):\n        values = source["season"].value_counts()\n        frame[label] = frame["season"].map(values).fillna(0).astype(int)\n    frame["relevant_fraction_of_eligible"] = np.where(\n        frame["eligible_storms"] > 0,\n        frame["relevant_events"] / frame["eligible_storms"],\n        np.nan,\n    )\n    frame["relevant_fraction_of_recurvers"] = np.where(\n        frame["recurving_storms"] > 0,\n        frame["relevant_events"] / frame["recurving_storms"],\n        np.nan,\n    )\n    return frame\n\n\ndef classification_overlap(first: pd.DataFrame, second: pd.DataFrame) -> dict:\n    a = set(first["sid"])\n    b = set(second["sid"])\n    union = a | b\n    return {\n        "first_n": len(a),\n        "second_n": len(b),\n        "overlap": len(a & b),\n        "only_first": len(a - b),\n        "only_second": len(b - a),\n        "jaccard": float(len(a & b) / len(union)) if union else np.nan,\n    }\n\n\ndef write_json(path: Path, value) -> None:\n    path.write_text(json.dumps(value, indent=2, default=str))\n\n\ndef config_dict(\n    detector: DetectorConfig = BASELINE_DETECTOR,\n    region: RegionConfig = BASELINE_REGION,\n) -> dict:\n    return {"detector": asdict(detector), "region": asdict(region)}\n\n\ndef suppress_expected_warnings() -> None:\n    warnings.filterwarnings("ignore", category=RuntimeWarning)\n'
path = PROJECT_ROOT / 'analysis_core.py'
path.write_text(source)
print(f"Wrote {path.name} ({len(source):,} characters)")

In [ ]:
from pathlib import Path

source = '"""Run the track-climatology analysis and generate manuscript outputs."""\n\nfrom __future__ import annotations\n\nimport math\nimport os\nfrom pathlib import Path\n\nimport cartopy\nimport cartopy.crs as ccrs\nimport cartopy.feature as cfeature\nfrom cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter\nimport matplotlib.pyplot as plt\nfrom matplotlib.lines import Line2D\nimport numpy as np\nimport pandas as pd\nfrom pyproj import CRS, Transformer\nfrom shapely.ops import transform as shapely_transform\nimport statsmodels.api as sm\nimport xarray as xr\n\nfrom analysis_core import (\n    BASELINE_DETECTOR,\n    BASELINE_REGION,\n    DetectorConfig,\n    YEAR_MAX,\n    YEAR_MIN,\n    SATELLITE_START,\n    annual_analysis_frame,\n    annual_counts,\n    binomial_pathway_trend,\n    build_track_cache,\n    classification_overlap,\n    classify_legacy_method,\n    classify_recurving_storms,\n    config_dict,\n    decode_text,\n    download_if_needed,\n    energy_distance_test,\n    load_ibtracs,\n    load_newfoundland_island_polygon,\n    lonlat_to_cartesian_km,\n    north_atlantic_indices,\n    poisson_trend,\n    post_track_distance_diagnostics_with_transformer,\n    quantile_distance_trend,\n    robust_linear_trend,\n    seasonal_counts,\n    status_summary,\n    tropical_origin_indices,\n    write_json,\n    IBTRACS_URL,\n)\n\n\nROOT = Path(os.environ.get("NL_RECURV_WORKDIR", Path.cwd())).resolve()\nDATA_PATH = Path(\n    os.environ.get("IBTRACS_PATH", ROOT / "data" / "IBTrACS.ALL.v04r00.nc")\n).resolve()\nOUTPUT_DIR = ROOT / "outputs"\nFIGURE_DIR = ROOT / "figures"\nCARTOPY_DIR = Path(\n    os.environ.get("CARTOPY_DATA_DIR", ROOT / "data" / "cartopy")\n).resolve()\n\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nFIGURE_DIR.mkdir(parents=True, exist_ok=True)\nCARTOPY_DIR.mkdir(parents=True, exist_ok=True)\ncartopy.config["data_dir"] = str(CARTOPY_DIR)\n\nBLUE = "#1769AA"\nORANGE = "#D97706"\nRED = "#B42318"\nGREEN = "#2E7D32"\nGRAY = "#555B66"\nLIGHT_BLUE = "#D8EAF7"\n\nplt.rcParams.update(\n    {\n        "figure.dpi": 120,\n        "savefig.dpi": 300,\n        "font.size": 10,\n        "axes.titlesize": 11,\n        "axes.labelsize": 10,\n        "legend.fontsize": 8.5,\n        "axes.spines.top": False,\n        "axes.spines.right": False,\n    }\n)\n\n\ndef save_figure(fig: plt.Figure, filename: str) -> None:\n    fig.tight_layout()\n    fig.savefig(FIGURE_DIR / filename, bbox_inches="tight", facecolor="white")\n    plt.close(fig)\n    print("Saved figure:", filename)\n\n\ndef format_interval(low: float, high: float, digits: int = 3) -> str:\n    return f"{low:.{digits}f}--{high:.{digits}f}"\n\n\ndef projection_axes(fig, position=111, extent=(-100, -40, 15, 60)):\n    ax = fig.add_subplot(position, projection=ccrs.PlateCarree())\n    ax.set_extent(extent, crs=ccrs.PlateCarree())\n    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#F1EFE8", zorder=0)\n    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="#EAF2F8", zorder=0)\n    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.55, zorder=2)\n    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.35, zorder=2)\n    longitude_ticks = np.arange(\n        math.ceil(extent[0] / 10.0) * 10.0,\n        math.floor(extent[1] / 10.0) * 10.0 + 0.1,\n        10.0,\n    )\n    latitude_ticks = np.arange(\n        math.ceil(extent[2] / 10.0) * 10.0,\n        math.floor(extent[3] / 10.0) * 10.0 + 0.1,\n        10.0,\n    )\n    ax.set_xticks(longitude_ticks, crs=ccrs.PlateCarree())\n    ax.set_yticks(latitude_ticks, crs=ccrs.PlateCarree())\n    ax.xaxis.set_major_formatter(LongitudeFormatter())\n    ax.yaxis.set_major_formatter(LatitudeFormatter())\n    ax.grid(linewidth=0.35, alpha=0.45, linestyle="--")\n    return ax\n\n\ndef dataframe_to_latex(\n    frame: pd.DataFrame,\n    path: Path,\n    caption: str,\n    label: str,\n    column_format: str | None = None,\n    resize_to_textwidth: bool = False,\n) -> None:\n    latex = frame.to_latex(\n        index=False,\n        escape=False,\n        column_format=column_format,\n        caption=caption,\n        label=label,\n        position="htbp",\n    )\n    if resize_to_textwidth:\n        latex = latex.replace(\n            "\\\\begin{tabular}",\n            "\\\\resizebox{\\\\textwidth}{!}{%\\n\\\\begin{tabular}",\n            1,\n        ).replace(\n            "\\\\end{tabular}",\n            "\\\\end{tabular}%\\n}",\n            1,\n        )\n    path.write_text(latex)\n\n\nprint("1/8 Loading IBTrACS")\ndownload_if_needed(IBTRACS_URL, DATA_PATH)\nvariables = [\n    "time",\n    "lat",\n    "lon",\n    "season",\n    "sid",\n    "basin",\n    "name",\n    "nature",\n    "usa_status",\n]\nds = load_ibtracs(DATA_PATH)[variables].load()\nnorth_atlantic_source_indices = north_atlantic_indices(ds)\nstorm_indices = tropical_origin_indices(ds, north_atlantic_source_indices)\nprint(\n    f"North Atlantic source storms in {YEAR_MIN}-{YEAR_MAX}: "\n    f"{len(north_atlantic_source_indices)}"\n)\nprint(\n    "Storms coded tropical at least once: "\n    f"{len(storm_indices)}"\n)\n\ntropical_origin_set = set(int(value) for value in storm_indices)\nsource_scope_audit = pd.DataFrame(\n    {\n        "dataset_index": north_atlantic_source_indices.astype(int),\n        "sid": [\n            decode_text(ds["sid"].values[int(k)])\n            for k in north_atlantic_source_indices\n        ],\n        "season": ds["season"].values[\n            north_atlantic_source_indices\n        ].astype(int),\n        "ever_coded_tropical": [\n            int(k) in tropical_origin_set\n            for k in north_atlantic_source_indices\n        ],\n    }\n)\n\n\nprint("2/8 Running the native-cadence workflow comparator")\nlegacy_events = classify_legacy_method(ds, storm_indices)\nlegacy_summary, _ = poisson_trend(annual_counts(legacy_events))\nif len(legacy_events) != 165:\n    raise RuntimeError(\n        "Native-cadence comparator validation failed: "\n        f"expected 165 after the tropical-origin filter, got {len(legacy_events)}"\n    )\n\n\nprint("3/8 Building regular 6-hour track archive")\ntrack_cache, time_audit = build_track_cache(ds, storm_indices, BASELINE_DETECTOR)\nisland_lonlat, island_projected = load_newfoundland_island_polygon(CARTOPY_DIR)\nvelocity_recurvers, eligible_storms = classify_recurving_storms(\n    ds,\n    storm_indices,\n    track_cache,\n    island_projected,\n    detector="velocity",\n)\nvelocity_events = velocity_recurvers[\n    velocity_recurvers["newfoundland_relevant"]\n].copy()\n\nheading_recurvers, _ = classify_recurving_storms(\n    ds,\n    storm_indices,\n    track_cache,\n    island_projected,\n    detector="heading",\n)\nheading_events = heading_recurvers[\n    heading_recurvers["newfoundland_relevant"]\n].copy()\n\nmethod_overlap = classification_overlap(velocity_events, heading_events)\nlegacy_overlap = classification_overlap(legacy_events, velocity_events)\n\nprint(\n    "Baseline events:",\n    len(velocity_events),\n    "| all baseline recurvers:",\n    len(velocity_recurvers),\n    "| eligible storms:",\n    int(eligible_storms["eligible"].sum()),\n)\n\n\nprint("4/8 Calculating frequency and pathway trends")\nannual = annual_analysis_frame(\n    velocity_events,\n    velocity_recurvers,\n    eligible_storms,\n)\nfull_count_summary, full_count_fitted = poisson_trend(\n    annual[["season", "relevant_events"]].rename(\n        columns={"relevant_events": "count"}\n    )\n)\nsatellite_annual = annual[annual["season"] >= SATELLITE_START].copy()\nsatellite_count_summary, satellite_count_fitted = poisson_trend(\n    satellite_annual[["season", "relevant_events"]].rename(\n        columns={"relevant_events": "count"}\n    )\n)\n\npathway_eligible_full = binomial_pathway_trend(\n    annual, "relevant_events", "eligible_storms"\n)\npathway_eligible_satellite = binomial_pathway_trend(\n    satellite_annual, "relevant_events", "eligible_storms"\n)\npathway_recurver_full = binomial_pathway_trend(\n    annual, "relevant_events", "recurving_storms"\n)\npathway_recurver_satellite = binomial_pathway_trend(\n    satellite_annual, "relevant_events", "recurving_storms"\n)\n\n\nprint("5/8 Calculating location, proximity, and seasonal diagnostics")\nlatitude_trend = robust_linear_trend(velocity_events, "recurv_lat")\nlongitude_trend = robust_linear_trend(velocity_events, "recurv_lon")\nearly_events = velocity_events[velocity_events["season"] < SATELLITE_START]\nlate_events = velocity_events[velocity_events["season"] >= SATELLITE_START]\nearly_location_xyz = lonlat_to_cartesian_km(\n    early_events["recurv_lon"].to_numpy(),\n    early_events["recurv_lat"].to_numpy(),\n)\nlate_location_xyz = lonlat_to_cartesian_km(\n    late_events["recurv_lon"].to_numpy(),\n    late_events["recurv_lat"].to_numpy(),\n)\nlocation_energy = energy_distance_test(\n    early_location_xyz,\n    late_location_xyz,\n)\n\nproximity_all_mean = robust_linear_trend(\n    velocity_recurvers, "minimum_distance_to_newfoundland_km"\n)\nproximity_all_median = quantile_distance_trend(\n    velocity_recurvers, "minimum_distance_to_newfoundland_km"\n)\nproximity_relevant_mean = robust_linear_trend(\n    velocity_events, "minimum_distance_to_newfoundland_km"\n)\nproximity_relevant_median = quantile_distance_trend(\n    velocity_events, "minimum_distance_to_newfoundland_km"\n)\n\nnon_endpoint_recurvers = velocity_recurvers[\n    ~velocity_recurvers["closest_at_track_endpoint"]\n].copy()\nproximity_non_endpoint_mean = robust_linear_trend(\n    non_endpoint_recurvers, "minimum_distance_to_newfoundland_km"\n)\nproximity_non_endpoint_median = quantile_distance_trend(\n    non_endpoint_recurvers, "minimum_distance_to_newfoundland_km"\n)\npost_duration_trend = robust_linear_trend(\n    velocity_recurvers, "post_recurvature_duration_hours"\n)\nearly_recurvers = velocity_recurvers[\n    velocity_recurvers["season"] < SATELLITE_START\n]\nlate_recurvers = velocity_recurvers[\n    velocity_recurvers["season"] >= SATELLITE_START\n]\nendpoint_summary = {\n    "endpoint_count": int(\n        velocity_recurvers["closest_at_track_endpoint"].sum()\n    ),\n    "endpoint_fraction": float(\n        velocity_recurvers["closest_at_track_endpoint"].mean()\n    ),\n    "relevant_endpoint_count": int(\n        velocity_events["closest_at_track_endpoint"].sum()\n    ),\n    "relevant_endpoint_fraction": float(\n        velocity_events["closest_at_track_endpoint"].mean()\n    ),\n    "early_endpoint_fraction": float(\n        early_recurvers["closest_at_track_endpoint"].mean()\n    ),\n    "late_endpoint_fraction": float(\n        late_recurvers["closest_at_track_endpoint"].mean()\n    ),\n    "non_endpoint_mean_trend": proximity_non_endpoint_mean,\n    "non_endpoint_median_trend": proximity_non_endpoint_median,\n    "post_recurvature_duration_trend": post_duration_trend,\n}\n\nseasonality = seasonal_counts(velocity_events)\nnature_counts = status_summary(velocity_events)\n\n\nprint("6/8 Running sensitivity analyses")\nthreshold_rows = []\nfor threshold in (300, 500, 600, 800, 1000):\n    events = velocity_recurvers[\n        velocity_recurvers["minimum_distance_to_newfoundland_km"] <= threshold\n    ]\n    trend, _ = poisson_trend(annual_counts(events))\n    satellite_events = events[events["season"] >= SATELLITE_START]\n    satellite_trend, _ = poisson_trend(\n        annual_counts(\n            satellite_events,\n            year_min=SATELLITE_START,\n            year_max=YEAR_MAX,\n        )\n    )\n    threshold_annual = annual_analysis_frame(\n        events,\n        velocity_recurvers,\n        eligible_storms,\n    )\n    threshold_satellite_annual = threshold_annual[\n        threshold_annual["season"] >= SATELLITE_START\n    ]\n    satellite_eligible_pathway = binomial_pathway_trend(\n        threshold_satellite_annual,\n        "relevant_events",\n        "eligible_storms",\n    )\n    satellite_recurver_pathway = binomial_pathway_trend(\n        threshold_satellite_annual,\n        "relevant_events",\n        "recurving_storms",\n    )\n    threshold_rows.append(\n        {\n            "threshold_km": threshold,\n            "events": len(events),\n            "irr_per_decade": trend["irr_per_decade"],\n            "ci_low": trend["ci_low"],\n            "ci_high": trend["ci_high"],\n            "p_value": trend["p_value"],\n            "dispersion": trend["pearson_dispersion"],\n            "satellite_events": len(satellite_events),\n            "satellite_irr_per_decade": satellite_trend["irr_per_decade"],\n            "satellite_ci_low": satellite_trend["ci_low"],\n            "satellite_ci_high": satellite_trend["ci_high"],\n            "satellite_p_value": satellite_trend["p_value"],\n            "satellite_hac_ci_low": satellite_trend["hac_ci_low"],\n            "satellite_hac_ci_high": satellite_trend["hac_ci_high"],\n            "satellite_hac_p_value": satellite_trend["hac_p_value"],\n            "satellite_eligible_or": satellite_eligible_pathway[\n                "odds_ratio_per_decade"\n            ],\n            "satellite_eligible_ci_low": satellite_eligible_pathway["ci_low"],\n            "satellite_eligible_ci_high": satellite_eligible_pathway["ci_high"],\n            "satellite_eligible_p_value": satellite_eligible_pathway["p_value"],\n            "satellite_recurver_or": satellite_recurver_pathway[\n                "odds_ratio_per_decade"\n            ],\n            "satellite_recurver_ci_low": satellite_recurver_pathway["ci_low"],\n            "satellite_recurver_ci_high": satellite_recurver_pathway["ci_high"],\n            "satellite_recurver_p_value": satellite_recurver_pathway["p_value"],\n        }\n    )\nthreshold_sensitivity = pd.DataFrame(threshold_rows)\n\ndetector_rows = []\nbaseline_set = set(velocity_events["sid"])\nfor latitude_gate in (25.0, 30.0):\n    for window_hours in (18, 24, 30):\n        for post_east in (2.5, 5.0, 7.5, 10.0):\n            detector_config = DetectorConfig(\n                grid_hours=6,\n                window_hours=window_hours,\n                latitude_gate_deg_n=latitude_gate,\n                pre_east_max_kmh=0.0,\n                post_east_min_kmh=post_east,\n                post_north_min_kmh=0.0,\n                min_eastward_acceleration_kmh=max(5.0, post_east),\n                maximum_source_gap_hours=12.0,\n            )\n            recurvers, candidate_eligible = classify_recurving_storms(\n                ds,\n                storm_indices,\n                track_cache,\n                island_projected,\n                detector="velocity",\n                detector_config=detector_config,\n            )\n            events = recurvers[recurvers["newfoundland_relevant"]]\n            trend, _ = poisson_trend(annual_counts(events))\n            satellite_events = events[events["season"] >= SATELLITE_START]\n            satellite_trend, _ = poisson_trend(\n                annual_counts(\n                    satellite_events,\n                    year_min=SATELLITE_START,\n                    year_max=YEAR_MAX,\n                )\n            )\n            candidate_annual = annual_analysis_frame(\n                events,\n                recurvers,\n                candidate_eligible,\n            )\n            candidate_satellite_annual = candidate_annual[\n                candidate_annual["season"] >= SATELLITE_START\n            ]\n            satellite_eligible_pathway = binomial_pathway_trend(\n                candidate_satellite_annual,\n                "relevant_events",\n                "eligible_storms",\n            )\n            satellite_recurver_pathway = binomial_pathway_trend(\n                candidate_satellite_annual,\n                "relevant_events",\n                "recurving_storms",\n            )\n            candidate_set = set(events["sid"])\n            union = baseline_set | candidate_set\n            detector_rows.append(\n                {\n                    "latitude_gate_deg_n": latitude_gate,\n                    "window_hours": window_hours,\n                    "post_east_min_kmh": post_east,\n                    "events": len(events),\n                    "irr_per_decade": trend["irr_per_decade"],\n                    "ci_low": trend["ci_low"],\n                    "ci_high": trend["ci_high"],\n                    "p_value": trend["p_value"],\n                    "dispersion": trend["pearson_dispersion"],\n                    "satellite_events": len(satellite_events),\n                    "satellite_irr_per_decade": satellite_trend[\n                        "irr_per_decade"\n                    ],\n                    "satellite_ci_low": satellite_trend["ci_low"],\n                    "satellite_ci_high": satellite_trend["ci_high"],\n                    "satellite_p_value": satellite_trend["p_value"],\n                    "satellite_hac_ci_low": satellite_trend["hac_ci_low"],\n                    "satellite_hac_ci_high": satellite_trend["hac_ci_high"],\n                    "satellite_hac_p_value": satellite_trend["hac_p_value"],\n                    "satellite_eligible_or": satellite_eligible_pathway[\n                        "odds_ratio_per_decade"\n                    ],\n                    "satellite_eligible_ci_low": satellite_eligible_pathway[\n                        "ci_low"\n                    ],\n                    "satellite_eligible_ci_high": satellite_eligible_pathway[\n                        "ci_high"\n                    ],\n                    "satellite_eligible_p_value": satellite_eligible_pathway[\n                        "p_value"\n                    ],\n                    "satellite_recurver_or": satellite_recurver_pathway[\n                        "odds_ratio_per_decade"\n                    ],\n                    "satellite_recurver_ci_low": satellite_recurver_pathway[\n                        "ci_low"\n                    ],\n                    "satellite_recurver_ci_high": satellite_recurver_pathway[\n                        "ci_high"\n                    ],\n                    "satellite_recurver_p_value": satellite_recurver_pathway[\n                        "p_value"\n                    ],\n                    "jaccard_with_baseline": (\n                        len(baseline_set & candidate_set) / len(union)\n                        if union\n                        else np.nan\n                    ),\n                }\n            )\ndetector_sensitivity = pd.DataFrame(detector_rows)\n\nalternative_crs = CRS.from_proj4(\n    "+proj=aeqd +lat_0=48.5 +lon_0=-56 "\n    "+datum=WGS84 +units=m +no_defs"\n)\nalternative_transformer = Transformer.from_crs(\n    "EPSG:4326",\n    alternative_crs,\n    always_xy=True,\n)\nalternative_island_projected = shapely_transform(\n    alternative_transformer.transform,\n    island_lonlat,\n)\nprojection_rows = []\nfor event in velocity_recurvers.itertuples(index=False):\n    alternative_diagnostics = (\n        post_track_distance_diagnostics_with_transformer(\n            track_cache[int(event.dataset_index)],\n            int(event.recurv_index),\n            alternative_island_projected,\n            alternative_transformer,\n        )\n    )\n    alternative_distance = alternative_diagnostics[\n        "minimum_distance_to_newfoundland_km"\n    ]\n    projection_rows.append(\n        {\n            "dataset_index": int(event.dataset_index),\n            "sid": event.sid,\n            "season": int(event.season),\n            "baseline_distance_km": float(\n                event.minimum_distance_to_newfoundland_km\n            ),\n            "alternative_distance_km": alternative_distance,\n            "absolute_difference_km": abs(\n                alternative_distance\n                - float(event.minimum_distance_to_newfoundland_km)\n            ),\n            "baseline_relevant": bool(event.newfoundland_relevant),\n            "alternative_relevant": bool(\n                alternative_distance\n                <= BASELINE_REGION.proximity_threshold_km\n            ),\n        }\n    )\nprojection_distance_sensitivity = pd.DataFrame(projection_rows)\nalternative_projection_events = projection_distance_sensitivity[\n    projection_distance_sensitivity["alternative_relevant"]\n]\nalternative_projection_count_trend, _ = poisson_trend(\n    annual_counts(alternative_projection_events)\n)\nalternative_projection_satellite_trend, _ = poisson_trend(\n    annual_counts(\n        alternative_projection_events[\n            alternative_projection_events["season"] >= SATELLITE_START\n        ],\n        year_min=SATELLITE_START,\n        year_max=YEAR_MAX,\n    )\n)\nalternative_projection_mean_trend = robust_linear_trend(\n    projection_distance_sensitivity,\n    "alternative_distance_km",\n)\nalternative_projection_median_trend = quantile_distance_trend(\n    projection_distance_sensitivity,\n    "alternative_distance_km",\n)\nprojection_summary = {\n    "baseline_projection": "EPSG:3347",\n    "alternative_projection": (\n        "Newfoundland-centred azimuthal equidistant "\n        "(48.5 N, 56 W; WGS84)"\n    ),\n    "baseline_relevant_events": int(len(velocity_events)),\n    "alternative_relevant_events": int(len(alternative_projection_events)),\n    "classification_changes": int(\n        (\n            projection_distance_sensitivity["baseline_relevant"]\n            != projection_distance_sensitivity["alternative_relevant"]\n        ).sum()\n    ),\n    "distance_correlation": float(\n        projection_distance_sensitivity[\n            ["baseline_distance_km", "alternative_distance_km"]\n        ].corr().iloc[0, 1]\n    ),\n    "median_absolute_difference_km": float(\n        projection_distance_sensitivity["absolute_difference_km"].median()\n    ),\n    "p95_absolute_difference_km": float(\n        projection_distance_sensitivity["absolute_difference_km"].quantile(\n            0.95\n        )\n    ),\n    "maximum_absolute_difference_km": float(\n        projection_distance_sensitivity["absolute_difference_km"].max()\n    ),\n    "full_count_trend": alternative_projection_count_trend,\n    "satellite_count_trend": alternative_projection_satellite_trend,\n    "mean_distance_trend": alternative_projection_mean_trend,\n    "median_distance_trend": alternative_projection_median_trend,\n}\n\nendpoint_distance_sensitivity = pd.DataFrame(\n    [\n        {\n            "sample": "All detected recurvers",\n            "n": len(velocity_recurvers),\n            "mean_slope_km_per_decade": proximity_all_mean[\n                "slope_per_decade"\n            ],\n            "mean_ci_low": proximity_all_mean["ci_low"],\n            "mean_ci_high": proximity_all_mean["ci_high"],\n            "mean_p_value": proximity_all_mean["p_value"],\n            "median_slope_km_per_decade": proximity_all_median[\n                "median_slope_km_per_decade"\n            ],\n            "median_ci_low": proximity_all_median["ci_low"],\n            "median_ci_high": proximity_all_median["ci_high"],\n            "median_p_value": proximity_all_median["p_value"],\n        },\n        {\n            "sample": "Excluding endpoint minima",\n            "n": len(non_endpoint_recurvers),\n            "mean_slope_km_per_decade": proximity_non_endpoint_mean[\n                "slope_per_decade"\n            ],\n            "mean_ci_low": proximity_non_endpoint_mean["ci_low"],\n            "mean_ci_high": proximity_non_endpoint_mean["ci_high"],\n            "mean_p_value": proximity_non_endpoint_mean["p_value"],\n            "median_slope_km_per_decade": proximity_non_endpoint_median[\n                "median_slope_km_per_decade"\n            ],\n            "median_ci_low": proximity_non_endpoint_median["ci_low"],\n            "median_ci_high": proximity_non_endpoint_median["ci_high"],\n            "median_p_value": proximity_non_endpoint_median["p_value"],\n        },\n    ]\n)\n\n\nprint("7/8 Writing data products")\nsource_scope_audit.to_csv(OUTPUT_DIR / "source_scope_audit.csv", index=False)\ntime_audit.to_csv(OUTPUT_DIR / "time_step_and_eligibility_audit.csv", index=False)\nlegacy_events.to_csv(\n    OUTPUT_DIR / "native_cadence_comparator_events.csv",\n    index=False,\n)\nvelocity_recurvers.to_csv(\n    OUTPUT_DIR / "all_corrected_recurving_storms.csv", index=False\n)\nvelocity_events.to_csv(\n    OUTPUT_DIR / "corrected_newfoundland_relevant_events.csv", index=False\n)\nheading_events.to_csv(\n    OUTPUT_DIR / "alternative_heading_events.csv", index=False\n)\neligible_storms.to_csv(OUTPUT_DIR / "eligible_storms.csv", index=False)\nannual.to_csv(OUTPUT_DIR / "annual_counts_and_denominators.csv", index=False)\nseasonality.to_csv(OUTPUT_DIR / "seasonality.csv", index=False)\nnature_counts.to_csv(OUTPUT_DIR / "nature_at_recurvature.csv", index=False)\nthreshold_sensitivity.to_csv(\n    OUTPUT_DIR / "threshold_sensitivity.csv", index=False\n)\ndetector_sensitivity.to_csv(\n    OUTPUT_DIR / "detector_sensitivity.csv", index=False\n)\nendpoint_distance_sensitivity.to_csv(\n    OUTPUT_DIR / "endpoint_distance_sensitivity.csv",\n    index=False,\n)\nprojection_distance_sensitivity.to_csv(\n    OUTPUT_DIR / "projection_distance_sensitivity.csv",\n    index=False,\n)\nwrite_json(\n    OUTPUT_DIR / "projection_sensitivity_summary.json",\n    projection_summary,\n)\n\nonly_velocity = sorted(set(velocity_events["sid"]) - set(heading_events["sid"]))\nonly_heading = sorted(set(heading_events["sid"]) - set(velocity_events["sid"]))\ndiscordant = pd.DataFrame(\n    [\n        {"sid": sid, "classification": "velocity_only"} for sid in only_velocity\n    ]\n    + [{"sid": sid, "classification": "heading_only"} for sid in only_heading]\n)\ndiscordant.to_csv(OUTPUT_DIR / "discordant_detector_events.csv", index=False)\n\n\nsummary = {\n    "configuration": config_dict(),\n    "sample": {\n        "north_atlantic_source_storms": int(\n            len(north_atlantic_source_indices)\n        ),\n        "tropical_origin_storms": int(len(storm_indices)),\n        "excluded_without_tropical_code": int(\n            len(north_atlantic_source_indices) - len(storm_indices)\n        ),\n        "eligible_storms": int(eligible_storms["eligible"].sum()),\n        "all_corrected_recurvers": int(len(velocity_recurvers)),\n        "corrected_relevant_events": int(len(velocity_events)),\n        "native_cadence_comparator_events": int(len(legacy_events)),\n        "heading_relevant_events": int(len(heading_events)),\n    },\n    "frequency": {\n        "full_period": full_count_summary,\n        "satellite_era": satellite_count_summary,\n    },\n    "pathway_rates": {\n        "relevant_among_eligible_full": pathway_eligible_full,\n        "relevant_among_eligible_satellite": pathway_eligible_satellite,\n        "relevant_among_recurvers_full": pathway_recurver_full,\n        "relevant_among_recurvers_satellite": pathway_recurver_satellite,\n    },\n    "location": {\n        "latitude": latitude_trend,\n        "longitude": longitude_trend,\n        "era_energy_test": location_energy,\n        "early_n": int(len(early_events)),\n        "late_n": int(len(late_events)),\n        "early_median_lat": float(early_events["recurv_lat"].median()),\n        "late_median_lat": float(late_events["recurv_lat"].median()),\n        "early_median_lon": float(early_events["recurv_lon"].median()),\n        "late_median_lon": float(late_events["recurv_lon"].median()),\n    },\n    "proximity": {\n        "all_recurvers_mean_trend": proximity_all_mean,\n        "all_recurvers_median_trend": proximity_all_median,\n        "relevant_only_mean_trend": proximity_relevant_mean,\n        "relevant_only_median_trend": proximity_relevant_median,\n        "all_recurvers_early_median_km": float(\n            velocity_recurvers.loc[\n                velocity_recurvers["season"] < SATELLITE_START,\n                "minimum_distance_to_newfoundland_km",\n            ].median()\n        ),\n        "all_recurvers_late_median_km": float(\n            velocity_recurvers.loc[\n                velocity_recurvers["season"] >= SATELLITE_START,\n                "minimum_distance_to_newfoundland_km",\n            ].median()\n        ),\n        "endpoint_diagnostic": endpoint_summary,\n        "alternative_projection": projection_summary,\n    },\n    "sensitivity": {\n        "threshold": {\n            "full_period_irr_range": [\n                float(threshold_sensitivity["irr_per_decade"].min()),\n                float(threshold_sensitivity["irr_per_decade"].max()),\n            ],\n            "satellite_irr_range": [\n                float(\n                    threshold_sensitivity[\n                        "satellite_irr_per_decade"\n                    ].min()\n                ),\n                float(\n                    threshold_sensitivity[\n                        "satellite_irr_per_decade"\n                    ].max()\n                ),\n            ],\n            "satellite_model_p_below_0_05": int(\n                (threshold_sensitivity["satellite_p_value"] < 0.05).sum()\n            ),\n            "satellite_hac_p_below_0_05": int(\n                (\n                    threshold_sensitivity["satellite_hac_p_value"]\n                    < 0.05\n                ).sum()\n            ),\n        },\n        "detector": {\n            "full_period_event_range": [\n                int(detector_sensitivity["events"].min()),\n                int(detector_sensitivity["events"].max()),\n            ],\n            "full_period_irr_range": [\n                float(detector_sensitivity["irr_per_decade"].min()),\n                float(detector_sensitivity["irr_per_decade"].max()),\n            ],\n            "satellite_event_range": [\n                int(detector_sensitivity["satellite_events"].min()),\n                int(detector_sensitivity["satellite_events"].max()),\n            ],\n            "satellite_irr_range": [\n                float(\n                    detector_sensitivity[\n                        "satellite_irr_per_decade"\n                    ].min()\n                ),\n                float(\n                    detector_sensitivity[\n                        "satellite_irr_per_decade"\n                    ].max()\n                ),\n            ],\n            "satellite_model_p_below_0_05": int(\n                (detector_sensitivity["satellite_p_value"] < 0.05).sum()\n            ),\n            "satellite_hac_p_below_0_05": int(\n                (\n                    detector_sensitivity["satellite_hac_p_value"]\n                    < 0.05\n                ).sum()\n            ),\n            "satellite_eligible_pathway_p_below_0_05": int(\n                (\n                    detector_sensitivity[\n                        "satellite_eligible_p_value"\n                    ]\n                    < 0.05\n                ).sum()\n            ),\n            "satellite_recurver_pathway_p_below_0_05": int(\n                (\n                    detector_sensitivity[\n                        "satellite_recurver_p_value"\n                    ]\n                    < 0.05\n                ).sum()\n            ),\n        },\n    },\n    "method_comparison": {\n        "velocity_vs_heading": method_overlap,\n        "legacy_vs_corrected": legacy_overlap,\n    },\n    "legacy_reproduction": legacy_summary,\n}\nwrite_json(OUTPUT_DIR / "analysis_summary.json", summary)\n\n\nprint("8/8 Generating figures and LaTeX tables")\n\n# Figure 1: objective geographic definition\ninverse_project = Transformer.from_crs(\n    "EPSG:3347", "EPSG:4326", always_xy=True\n).transform\nbuffer_600_lonlat = shapely_transform(\n    inverse_project, island_projected.buffer(600_000.0)\n)\nfig = plt.figure(figsize=(7.4, 6.2))\nax = projection_axes(fig, extent=(-70, -44, 40, 57))\nax.add_geometries(\n    [buffer_600_lonlat],\n    crs=ccrs.PlateCarree(),\n    facecolor=LIGHT_BLUE,\n    edgecolor=BLUE,\n    linewidth=1.0,\n    alpha=0.55,\n    zorder=1,\n)\nax.add_geometries(\n    [island_lonlat],\n    crs=ccrs.PlateCarree(),\n    facecolor="#F7F2D0",\n    edgecolor="#222222",\n    linewidth=1.0,\n    zorder=3,\n)\nax.set_title("Newfoundland island and the 600-km track-proximity region")\nax.legend(\n    handles=[\n        Line2D([0], [0], color=BLUE, lw=6, alpha=0.35, label="600-km buffer"),\n        Line2D([0], [0], color="#222222", lw=2, label="Newfoundland island"),\n    ],\n    loc="lower left",\n)\nsave_figure(fig, "fig1_newfoundland_geometry.png")\n\n# Figure 2: seasonality\nfig, ax = plt.subplots(figsize=(7.4, 4.2))\nax.bar(\n    seasonality["month_name"].str[:3],\n    seasonality["events"],\n    color=BLUE,\n    width=0.78,\n)\nax.set_ylabel("Number of events")\nax.set_xlabel("Month of diagnosed recurvature")\nax.set_title(\n    f"Seasonality of baseline Newfoundland-relevant recurvature "\n    f"({YEAR_MIN}-{YEAR_MAX})"\n)\nax.grid(axis="y", alpha=0.2)\nsave_figure(fig, "fig2_seasonality.png")\n\n# Figure 3: annual counts with Poisson fit\nfig, ax = plt.subplots(figsize=(8.2, 4.5))\nax.plot(\n    annual["season"],\n    annual["relevant_events"],\n    color=GRAY,\n    lw=0.9,\n    marker="o",\n    ms=2.8,\n    label="Annual count",\n)\nax.fill_between(\n    full_count_fitted["season"],\n    full_count_fitted["ci_low"],\n    full_count_fitted["ci_high"],\n    color=BLUE,\n    alpha=0.16,\n    linewidth=0,\n    label="95% confidence interval",\n)\nax.plot(\n    full_count_fitted["season"],\n    full_count_fitted["fitted"],\n    color=BLUE,\n    lw=2.0,\n    label="Poisson fitted mean",\n)\nax.axvline(\n    SATELLITE_START,\n    color=ORANGE,\n    linestyle="--",\n    lw=1,\n    label="Satellite-era sensitivity start",\n)\nax.set_ylabel("Number of events")\nax.set_xlabel("Season")\nax.set_ylim(bottom=0)\nax.set_title("Annual Newfoundland-relevant recurvature counts")\nax.legend(ncol=2, frameon=False)\nax.grid(axis="y", alpha=0.2)\nsave_figure(fig, "fig3_annual_counts_poisson.png")\n\n# Figure 4: rolling context, with partial-window ends suppressed\nrolling = (\n    annual.set_index("season")["relevant_events"]\n    .rolling(window=10, center=True, min_periods=10)\n    .mean()\n)\nfig, ax = plt.subplots(figsize=(8.2, 4.4))\nax.plot(\n    annual["season"],\n    annual["relevant_events"],\n    color="#8A8F98",\n    lw=0.8,\n    label="Annual count",\n)\nax.plot(\n    rolling.index,\n    rolling.values,\n    color=ORANGE,\n    lw=2.1,\n    label="Centered 10-year mean",\n)\nax.set_ylabel("Number of events")\nax.set_xlabel("Season")\nax.set_title("Interannual variability and centered 10-year mean")\nax.legend(frameon=False)\nax.grid(axis="y", alpha=0.2)\nsave_figure(fig, "fig4_rolling_mean.png")\n\n# Figure 5: recurvature locations in equal-area visual panels\nfig = plt.figure(figsize=(10.2, 4.8))\nfor panel, events, title in (\n    (121, early_events, f"{YEAR_MIN}-{SATELLITE_START - 1} (N={len(early_events)})"),\n    (122, late_events, f"{SATELLITE_START}-{YEAR_MAX} (N={len(late_events)})"),\n):\n    ax = projection_axes(fig, panel, extent=(-100, -40, 20, 55))\n    ax.scatter(\n        events["recurv_lon"],\n        events["recurv_lat"],\n        s=15,\n        alpha=0.65,\n        color=BLUE if panel == 121 else ORANGE,\n        transform=ccrs.PlateCarree(),\n        zorder=4,\n    )\n    ax.scatter(\n        events["recurv_lon"].median(),\n        events["recurv_lat"].median(),\n        marker="*",\n        s=95,\n        color=RED,\n        edgecolor="white",\n        linewidth=0.5,\n        transform=ccrs.PlateCarree(),\n        zorder=5,\n        label="Median location",\n    )\n    ax.set_title(title)\n    ax.legend(loc="lower left", frameon=False)\nfig.suptitle("Baseline recurvature-point locations by era", y=1.01)\nsave_figure(fig, "fig5_recurvature_locations.png")\n\n# Figure 6: proximity of every detected recurver, avoiding threshold truncation\ndistance_response = velocity_recurvers[\n    "minimum_distance_to_newfoundland_km"\n].to_numpy(dtype=float)\nyear = velocity_recurvers["season"].to_numpy(dtype=float)\ndesign = sm.add_constant(year - year.mean())\ndistance_model = sm.OLS(distance_response, design).fit(cov_type="HC3")\ngrid_year = np.linspace(YEAR_MIN, YEAR_MAX, 150)\ngrid_design = sm.add_constant(grid_year - year.mean())\ngrid_prediction = distance_model.get_prediction(grid_design).summary_frame()\n\nfig, ax = plt.subplots(figsize=(8.2, 4.8))\ninside = velocity_recurvers["newfoundland_relevant"]\nax.scatter(\n    velocity_recurvers.loc[~inside, "season"],\n    velocity_recurvers.loc[\n        ~inside, "minimum_distance_to_newfoundland_km"\n    ],\n    s=10,\n    alpha=0.35,\n    color="#8A8F98",\n    label="Other recurvers",\n)\nax.scatter(\n    velocity_recurvers.loc[inside, "season"],\n    velocity_recurvers.loc[\n        inside, "minimum_distance_to_newfoundland_km"\n    ],\n    s=13,\n    alpha=0.65,\n    color=BLUE,\n    label="Within 600 km",\n)\nax.plot(grid_year, grid_prediction["mean"], color=ORANGE, lw=2, label="Mean trend")\nax.fill_between(\n    grid_year,\n    grid_prediction["mean_ci_lower"],\n    grid_prediction["mean_ci_upper"],\n    color=ORANGE,\n    alpha=0.15,\n    linewidth=0,\n)\nax.axhline(\n    BASELINE_REGION.proximity_threshold_km,\n    color=RED,\n    linestyle="--",\n    lw=1,\n    label="600-km classification threshold",\n)\nax.set_ylabel("Minimum post-recurvature distance (km)")\nax.set_xlabel("Season")\nax.set_title("Proximity of all detected recurving storms to Newfoundland")\nax.legend(ncol=2, frameon=False)\nax.grid(axis="y", alpha=0.2)\nsave_figure(fig, "fig6_proximity_all_recurvers.png")\n\n# Figure 7: normalized distance distributions by era\nfig, ax = plt.subplots(figsize=(7.6, 4.6))\nfor events, color, label in (\n    (\n        velocity_recurvers[velocity_recurvers["season"] < SATELLITE_START],\n        BLUE,\n        f"{YEAR_MIN}-{SATELLITE_START - 1}",\n    ),\n    (\n        velocity_recurvers[velocity_recurvers["season"] >= SATELLITE_START],\n        ORANGE,\n        f"{SATELLITE_START}-{YEAR_MAX}",\n    ),\n):\n    values = np.sort(\n        events["minimum_distance_to_newfoundland_km"].to_numpy(dtype=float)\n    )\n    probability = np.arange(1, len(values) + 1) / len(values)\n    ax.step(values, probability, where="post", color=color, lw=2, label=label)\nax.axvline(\n    BASELINE_REGION.proximity_threshold_km,\n    color=RED,\n    linestyle="--",\n    lw=1,\n    label="600-km threshold",\n)\nax.set_xlabel("Minimum post-recurvature distance (km)")\nax.set_ylabel("Cumulative proportion")\nax.set_title("Normalized proximity distributions for all recurvers")\nax.legend(frameon=False)\nax.grid(alpha=0.2)\nsave_figure(fig, "fig7_proximity_ecdf.png")\n\n# Figure 8: threshold sensitivity distinguishes event magnitude from trend\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 4.2))\nax1.plot(\n    threshold_sensitivity["threshold_km"],\n    threshold_sensitivity["events"],\n    marker="o",\n    color=BLUE,\n    lw=2,\n)\nax1.axvline(600, color=RED, linestyle="--", lw=1)\nax1.set_xlabel("Proximity threshold (km)")\nax1.set_ylabel("Classified events")\nax1.set_title("Event-list sensitivity")\nax1.grid(alpha=0.2)\n\nax2.errorbar(\n    threshold_sensitivity["threshold_km"],\n    threshold_sensitivity["irr_per_decade"],\n    yerr=[\n        threshold_sensitivity["irr_per_decade"]\n        - threshold_sensitivity["ci_low"],\n        threshold_sensitivity["ci_high"]\n        - threshold_sensitivity["irr_per_decade"],\n    ],\n    fmt="o-",\n    color=ORANGE,\n    capsize=3,\n    lw=1.8,\n)\nax2.axhline(1.0, color=GRAY, linestyle="--", lw=1)\nax2.axvline(600, color=RED, linestyle="--", lw=1)\nax2.set_xlabel("Proximity threshold (km)")\nax2.set_ylabel("IRR per decade (95% CI)")\nax2.set_title("Frequency-trend sensitivity")\nax2.grid(alpha=0.2)\nsave_figure(fig, "fig8_threshold_sensitivity.png")\n\n# Appendix figure: all detector-discordant tracks\nvelocity_lookup = {\n    row["sid"]: row for _, row in velocity_events.iterrows()\n}\nheading_lookup = {row["sid"]: row for _, row in heading_events.iterrows()}\ndiscordant_ids = sorted(set(velocity_lookup) ^ set(heading_lookup))\nncols = 4\nnrows = int(np.ceil(len(discordant_ids) / ncols))\nfig, axes = plt.subplots(nrows, ncols, figsize=(10.5, 2.45 * nrows))\naxes = np.atleast_1d(axes).ravel()\nisland_parts = (\n    list(island_lonlat.geoms)\n    if hasattr(island_lonlat, "geoms")\n    else [island_lonlat]\n)\nfor ax, sid in zip(axes, discordant_ids):\n    row = velocity_lookup.get(sid, heading_lookup.get(sid))\n    track = track_cache[int(row["dataset_index"])]\n    ax.plot(track["lon"], track["lat"], color="#9A9A9A", lw=0.9)\n    for polygon in island_parts:\n        x_coord, y_coord = polygon.exterior.xy\n        ax.fill(\n            x_coord,\n            y_coord,\n            facecolor="#F1EFE8",\n            edgecolor="#333333",\n            linewidth=0.45,\n        )\n    if sid in velocity_lookup:\n        event = velocity_lookup[sid]\n        ax.scatter(\n            event["recurv_lon"],\n            event["recurv_lat"],\n            color=RED,\n            s=20,\n            label="Velocity detector",\n        )\n    if sid in heading_lookup:\n        event = heading_lookup[sid]\n        ax.scatter(\n            event["recurv_lon"],\n            event["recurv_lat"],\n            color=BLUE,\n            marker="x",\n            s=22,\n            label="Heading detector",\n        )\n    detector_label = "V" if sid in velocity_lookup else "H"\n    ax.set_title(\n        f"{row[\'name\']} ({int(row[\'season\'])}; {detector_label} only)",\n        fontsize=7.5,\n    )\n    ax.grid(alpha=0.18)\nfor ax in axes[len(discordant_ids) :]:\n    ax.axis("off")\nfig.suptitle("Tracks classified by only one detector", y=1.002)\nsave_figure(fig, "figA1_detector_discordant_tracks.pdf")\nstale_discordant_png = FIGURE_DIR / "figA1_detector_discordant_tracks.png"\nif stale_discordant_png.exists():\n    stale_discordant_png.unlink()\n\n\n# Main trend table\nfrequency_table = pd.DataFrame(\n    [\n        {\n            "Period": f"{YEAR_MIN}--{YEAR_MAX}",\n            "$N$": full_count_summary["total_events"],\n            "IRR decade$^{-1}$": f"{full_count_summary[\'irr_per_decade\']:.3f}",\n            "95\\\\% CI": format_interval(\n                full_count_summary["ci_low"], full_count_summary["ci_high"]\n            ),\n            "$p$": f"{full_count_summary[\'p_value\']:.3f}",\n            "HAC 95\\\\% CI": format_interval(\n                full_count_summary["hac_ci_low"],\n                full_count_summary["hac_ci_high"],\n            ),\n            "Dispersion": f"{full_count_summary[\'pearson_dispersion\']:.2f}",\n        },\n        {\n            "Period": f"{SATELLITE_START}--{YEAR_MAX}",\n            "$N$": satellite_count_summary["total_events"],\n            "IRR decade$^{-1}$": f"{satellite_count_summary[\'irr_per_decade\']:.3f}",\n            "95\\\\% CI": format_interval(\n                satellite_count_summary["ci_low"],\n                satellite_count_summary["ci_high"],\n            ),\n            "$p$": f"{satellite_count_summary[\'p_value\']:.3f}",\n            "HAC 95\\\\% CI": format_interval(\n                satellite_count_summary["hac_ci_low"],\n                satellite_count_summary["hac_ci_high"],\n            ),\n            "Dispersion": f"{satellite_count_summary[\'pearson_dispersion\']:.2f}",\n        },\n    ]\n)\ndataframe_to_latex(\n    frequency_table,\n    OUTPUT_DIR / "table_frequency_trends.tex",\n    (\n        "Poisson trend estimates for annual Newfoundland-relevant "\n        "recurvature counts. HAC intervals allow for short-lag residual "\n        "dependence."\n    ),\n    "tab:frequency_trends",\n)\n\npathway_table = pd.DataFrame(\n    [\n        {\n            "Denominator": "Eligible storms",\n            "Period": f"{YEAR_MIN}--{YEAR_MAX}",\n            "Events/trials": (\n                f"{pathway_eligible_full[\'total_successes\']}/"\n                f"{pathway_eligible_full[\'total_trials\']}"\n            ),\n            "OR decade$^{-1}$": f"{pathway_eligible_full[\'odds_ratio_per_decade\']:.3f}",\n            "95\\\\% CI": format_interval(\n                pathway_eligible_full["ci_low"],\n                pathway_eligible_full["ci_high"],\n            ),\n            "$p$": f"{pathway_eligible_full[\'p_value\']:.3f}",\n        },\n        {\n            "Denominator": "Eligible storms",\n            "Period": f"{SATELLITE_START}--{YEAR_MAX}",\n            "Events/trials": (\n                f"{pathway_eligible_satellite[\'total_successes\']}/"\n                f"{pathway_eligible_satellite[\'total_trials\']}"\n            ),\n            "OR decade$^{-1}$": (\n                f"{pathway_eligible_satellite[\'odds_ratio_per_decade\']:.3f}"\n            ),\n            "95\\\\% CI": format_interval(\n                pathway_eligible_satellite["ci_low"],\n                pathway_eligible_satellite["ci_high"],\n            ),\n            "$p$": f"{pathway_eligible_satellite[\'p_value\']:.3f}",\n        },\n        {\n            "Denominator": "Detected recurvers",\n            "Period": f"{YEAR_MIN}--{YEAR_MAX}",\n            "Events/trials": (\n                f"{pathway_recurver_full[\'total_successes\']}/"\n                f"{pathway_recurver_full[\'total_trials\']}"\n            ),\n            "OR decade$^{-1}$": f"{pathway_recurver_full[\'odds_ratio_per_decade\']:.3f}",\n            "95\\\\% CI": format_interval(\n                pathway_recurver_full["ci_low"],\n                pathway_recurver_full["ci_high"],\n            ),\n            "$p$": f"{pathway_recurver_full[\'p_value\']:.3f}",\n        },\n        {\n            "Denominator": "Detected recurvers",\n            "Period": f"{SATELLITE_START}--{YEAR_MAX}",\n            "Events/trials": (\n                f"{pathway_recurver_satellite[\'total_successes\']}/"\n                f"{pathway_recurver_satellite[\'total_trials\']}"\n            ),\n            "OR decade$^{-1}$": (\n                f"{pathway_recurver_satellite[\'odds_ratio_per_decade\']:.3f}"\n            ),\n            "95\\\\% CI": format_interval(\n                pathway_recurver_satellite["ci_low"],\n                pathway_recurver_satellite["ci_high"],\n            ),\n            "$p$": f"{pathway_recurver_satellite[\'p_value\']:.3f}",\n        },\n    ]\n)\ndataframe_to_latex(\n    pathway_table,\n    OUTPUT_DIR / "table_pathway_trends.tex",\n    (\n        "Binomial trends in the probability of a Newfoundland-relevant "\n        "pathway among eligible storms and among detected recurvers."\n    ),\n    "tab:pathway_trends",\n)\n\nlocation_proximity_table = pd.DataFrame(\n    [\n        {\n            "Outcome": "Recurvature latitude",\n            "Sample": "Relevant events",\n            "Effect decade$^{-1}$": f"{latitude_trend[\'slope_per_decade\']:.2f}$^\\\\circ$",\n            "95\\\\% CI": (\n                f"{latitude_trend[\'ci_low\']:.2f}--"\n                f"{latitude_trend[\'ci_high\']:.2f}$^\\\\circ$"\n            ),\n            "$p$": f"{latitude_trend[\'p_value\']:.3f}",\n        },\n        {\n            "Outcome": "Recurvature longitude",\n            "Sample": "Relevant events",\n            "Effect decade$^{-1}$": f"{longitude_trend[\'slope_per_decade\']:.2f}$^\\\\circ$",\n            "95\\\\% CI": (\n                f"{longitude_trend[\'ci_low\']:.2f}--"\n                f"{longitude_trend[\'ci_high\']:.2f}$^\\\\circ$"\n            ),\n            "$p$": f"{longitude_trend[\'p_value\']:.3f}",\n        },\n        {\n            "Outcome": "Mean minimum distance",\n            "Sample": "All recurvers",\n            "Effect decade$^{-1}$": (\n                f"{proximity_all_mean[\'slope_per_decade\']:.1f} km"\n            ),\n            "95\\\\% CI": (\n                f"{proximity_all_mean[\'ci_low\']:.1f}--"\n                f"{proximity_all_mean[\'ci_high\']:.1f} km"\n            ),\n            "$p$": f"{proximity_all_mean[\'p_value\']:.3f}",\n        },\n        {\n            "Outcome": "Median minimum distance",\n            "Sample": "All recurvers",\n            "Effect decade$^{-1}$": (\n                f"{proximity_all_median[\'median_slope_km_per_decade\']:.1f} km"\n            ),\n            "95\\\\% CI": (\n                f"{proximity_all_median[\'ci_low\']:.1f}--"\n                f"{proximity_all_median[\'ci_high\']:.1f} km"\n            ),\n            "$p$": f"{proximity_all_median[\'p_value\']:.3f}",\n        },\n    ]\n)\ndataframe_to_latex(\n    location_proximity_table,\n    OUTPUT_DIR / "table_location_proximity.tex",\n    (\n        "Trend estimates for recurvature-point location and minimum "\n        "post-recurvature distance. Linear estimates use HC3 standard "\n        "errors; the median-distance estimate uses quantile regression."\n    ),\n    "tab:location_proximity",\n)\n\nthreshold_table = threshold_sensitivity[\n    [\n        "threshold_km",\n        "events",\n        "irr_per_decade",\n        "ci_low",\n        "ci_high",\n        "p_value",\n        "dispersion",\n    ]\n].copy()\nthreshold_table.columns = [\n    "Threshold (km)",\n    "$N$",\n    "IRR decade$^{-1}$",\n    "CI low",\n    "CI high",\n    "$p$",\n    "Dispersion",\n]\nfor column in ("IRR decade$^{-1}$", "CI low", "CI high", "$p$", "Dispersion"):\n    threshold_table[column] = threshold_table[column].map(lambda value: f"{value:.3f}")\ndataframe_to_latex(\n    threshold_table,\n    OUTPUT_DIR / "table_threshold_sensitivity.tex",\n    (\n        "Sensitivity of event classification and Poisson trend estimates "\n        "to the Newfoundland-island proximity threshold."\n    ),\n    "tab:threshold_sensitivity",\n)\n\nsatellite_threshold_table = pd.DataFrame(\n    {\n        "Threshold (km)": threshold_sensitivity["threshold_km"].astype(int),\n        "$N$": threshold_sensitivity["satellite_events"].astype(int),\n        "Count IRR (95\\\\% CI)": [\n            (\n                f"{row.satellite_irr_per_decade:.3f} "\n                f"({row.satellite_ci_low:.3f}--"\n                f"{row.satellite_ci_high:.3f})"\n            )\n            for row in threshold_sensitivity.itertuples(index=False)\n        ],\n        "$p$": threshold_sensitivity["satellite_p_value"].map(\n            lambda value: f"{value:.3f}"\n        ),\n        "Eligible-storm OR (95\\\\% CI)": [\n            (\n                f"{row.satellite_eligible_or:.3f} "\n                f"({row.satellite_eligible_ci_low:.3f}--"\n                f"{row.satellite_eligible_ci_high:.3f})"\n            )\n            for row in threshold_sensitivity.itertuples(index=False)\n        ],\n        "Recurver OR (95\\\\% CI)": [\n            (\n                f"{row.satellite_recurver_or:.3f} "\n                f"({row.satellite_recurver_ci_low:.3f}--"\n                f"{row.satellite_recurver_ci_high:.3f})"\n            )\n            for row in threshold_sensitivity.itertuples(index=False)\n        ],\n    }\n)\ndataframe_to_latex(\n    satellite_threshold_table,\n    OUTPUT_DIR / "table_satellite_threshold_sensitivity.tex",\n    (\n        "Satellite-era sensitivity to the Newfoundland-island proximity "\n        "threshold. Conditional odds ratios use the eligible-storm and "\n        "detected-recurver denominators."\n    ),\n    "tab:satellite_threshold_sensitivity",\n    resize_to_textwidth=True,\n)\n\nendpoint_table = pd.DataFrame(\n    {\n        "Sample": endpoint_distance_sensitivity["sample"],\n        "$N$": endpoint_distance_sensitivity["n"].astype(int),\n        "Mean slope (95\\\\% CI)": [\n            (\n                f"{row.mean_slope_km_per_decade:.1f} "\n                f"({row.mean_ci_low:.1f}--{row.mean_ci_high:.1f})"\n            )\n            for row in endpoint_distance_sensitivity.itertuples(index=False)\n        ],\n        "Mean $p$": endpoint_distance_sensitivity["mean_p_value"].map(\n            lambda value: f"{value:.3f}"\n        ),\n        "Median slope (95\\\\% CI)": [\n            (\n                f"{row.median_slope_km_per_decade:.1f} "\n                f"({row.median_ci_low:.1f}--{row.median_ci_high:.1f})"\n            )\n            for row in endpoint_distance_sensitivity.itertuples(index=False)\n        ],\n        "Median $p$": endpoint_distance_sensitivity[\n            "median_p_value"\n        ].map(lambda value: f"{value:.3f}"),\n    }\n)\ndataframe_to_latex(\n    endpoint_table,\n    OUTPUT_DIR / "table_endpoint_sensitivity.tex",\n    (\n        "Sensitivity of minimum-distance trends to post-recurvature tracks "\n        "whose closest observed location is the final recorded point. "\n        "Slopes are in km decade$^{-1}$."\n    ),\n    "tab:endpoint_sensitivity",\n    resize_to_textwidth=True,\n)\n\nmethod_table = pd.DataFrame(\n    [\n        {\n            "Comparison": "Velocity vs. heading",\n            "$N_1$": method_overlap["first_n"],\n            "$N_2$": method_overlap["second_n"],\n            "Both": method_overlap["overlap"],\n            "Only 1": method_overlap["only_first"],\n            "Only 2": method_overlap["only_second"],\n            "$J$": f"{method_overlap[\'jaccard\']:.3f}",\n        },\n    ]\n)\ndataframe_to_latex(\n    method_table,\n    OUTPUT_DIR / "table_method_comparison.tex",\n    "Event-list overlap between the baseline velocity and alternative heading detectors.",\n    "tab:method_comparison",\n)\n\n\npeak = seasonality.loc[seasonality["events"].idxmax()]\naug_oct = int(\n    seasonality.loc[seasonality["month"].isin([8, 9, 10]), "events"].sum()\n)\naug_oct_pct = 100.0 * aug_oct / len(velocity_events)\nts_events = int(\n    nature_counts.loc[nature_counts["nature_code"] == "TS", "events"].sum()\n)\nthreshold_300 = threshold_sensitivity.loc[\n    threshold_sensitivity["threshold_km"] == 300\n].iloc[0]\nthreshold_1000 = threshold_sensitivity.loc[\n    threshold_sensitivity["threshold_km"] == 1000\n].iloc[0]\nmost_detectable_satellite_detector = detector_sensitivity.sort_values(\n    "satellite_p_value"\n).iloc[0]\nfull_record_decades = (YEAR_MAX - YEAR_MIN) / 10.0\nsatellite_record_decades = (YEAR_MAX - SATELLITE_START) / 10.0\n\nmacro_values = {\n    "NorthAtlanticSourceN": len(north_atlantic_source_indices),\n    "TropicalOriginSourceN": len(storm_indices),\n    "NonTropicalExcludedN": (\n        len(north_atlantic_source_indices) - len(storm_indices)\n    ),\n    "LegacyN": len(legacy_events),\n    "BaselineN": len(velocity_events),\n    "AllRecurverN": len(velocity_recurvers),\n    "EligibleN": int(eligible_storms["eligible"].sum()),\n    "HeadingN": len(heading_events),\n    "DetectorOverlap": method_overlap["overlap"],\n    "DetectorOnlyVelocity": method_overlap["only_first"],\n    "DetectorOnlyHeading": method_overlap["only_second"],\n    "DetectorDiscordantN": (\n        method_overlap["only_first"] + method_overlap["only_second"]\n    ),\n    "DetectorJaccard": f"{method_overlap[\'jaccard\']:.3f}",\n    "LegacyOverlap": legacy_overlap["overlap"],\n    "LegacyOnly": legacy_overlap["only_first"],\n    "BaselineOnlyLegacyComparison": legacy_overlap["only_second"],\n    "LegacyJaccard": f"{legacy_overlap[\'jaccard\']:.3f}",\n    "FullIRR": f"{full_count_summary[\'irr_per_decade\']:.3f}",\n    "FullCILow": f"{full_count_summary[\'ci_low\']:.3f}",\n    "FullCIHigh": f"{full_count_summary[\'ci_high\']:.3f}",\n    "FullP": f"{full_count_summary[\'p_value\']:.3f}",\n    "FullDispersion": f"{full_count_summary[\'pearson_dispersion\']:.2f}",\n    "FullHACCILow": f"{full_count_summary[\'hac_ci_low\']:.3f}",\n    "FullHACCIHigh": f"{full_count_summary[\'hac_ci_high\']:.3f}",\n    "FullHACP": f"{full_count_summary[\'hac_p_value\']:.3f}",\n    "FullLagOne": (\n        f"{full_count_summary[\'lag1_residual_autocorrelation\']:.3f}"\n    ),\n    "FullLjungBoxP": f"{full_count_summary[\'ljung_box_p_value\']:.3f}",\n    "FullRecordFactorLow": (\n        f"{full_count_summary[\'ci_low\'] ** full_record_decades:.2f}"\n    ),\n    "FullRecordFactorHigh": (\n        f"{full_count_summary[\'ci_high\'] ** full_record_decades:.2f}"\n    ),\n    "SatelliteN": satellite_count_summary["total_events"],\n    "SatelliteIRR": f"{satellite_count_summary[\'irr_per_decade\']:.3f}",\n    "SatelliteCILow": f"{satellite_count_summary[\'ci_low\']:.3f}",\n    "SatelliteCIHigh": f"{satellite_count_summary[\'ci_high\']:.3f}",\n    "SatelliteP": f"{satellite_count_summary[\'p_value\']:.3f}",\n    "SatelliteDispersion": f"{satellite_count_summary[\'pearson_dispersion\']:.2f}",\n    "SatelliteHACCILow": f"{satellite_count_summary[\'hac_ci_low\']:.3f}",\n    "SatelliteHACCIHigh": f"{satellite_count_summary[\'hac_ci_high\']:.3f}",\n    "SatelliteHACP": f"{satellite_count_summary[\'hac_p_value\']:.3f}",\n    "SatelliteRecordFactorLow": (\n        f"{satellite_count_summary[\'ci_low\'] ** satellite_record_decades:.2f}"\n    ),\n    "SatelliteRecordFactorHigh": (\n        f"{satellite_count_summary[\'ci_high\'] ** satellite_record_decades:.2f}"\n    ),\n    "EligibleOR": f"{pathway_eligible_full[\'odds_ratio_per_decade\']:.3f}",\n    "EligibleORLow": f"{pathway_eligible_full[\'ci_low\']:.3f}",\n    "EligibleORHigh": f"{pathway_eligible_full[\'ci_high\']:.3f}",\n    "EligibleORP": f"{pathway_eligible_full[\'p_value\']:.3f}",\n    "RecurverOR": f"{pathway_recurver_full[\'odds_ratio_per_decade\']:.3f}",\n    "RecurverORLow": f"{pathway_recurver_full[\'ci_low\']:.3f}",\n    "RecurverORHigh": f"{pathway_recurver_full[\'ci_high\']:.3f}",\n    "RecurverORP": f"{pathway_recurver_full[\'p_value\']:.3f}",\n    "PeakMonth": str(peak["month_name"]),\n    "PeakMonthN": int(peak["events"]),\n    "PeakMonthPct": f"{peak[\'percent\']:.1f}",\n    "AugOctN": aug_oct,\n    "AugOctPct": f"{aug_oct_pct:.1f}",\n    "EarlyN": len(early_events),\n    "LateN": len(late_events),\n    "EarlyMedianLatitude": f"{early_events[\'recurv_lat\'].median():.1f}",\n    "LateMedianLatitude": f"{late_events[\'recurv_lat\'].median():.1f}",\n    "EarlyMedianLongitudeWest": (\n        f"{abs(early_events[\'recurv_lon\'].median()):.1f}"\n    ),\n    "LateMedianLongitudeWest": (\n        f"{abs(late_events[\'recurv_lon\'].median()):.1f}"\n    ),\n    "LatitudeSlope": f"{latitude_trend[\'slope_per_decade\']:.2f}",\n    "LatitudeLow": f"{latitude_trend[\'ci_low\']:.2f}",\n    "LatitudeHigh": f"{latitude_trend[\'ci_high\']:.2f}",\n    "LatitudeP": f"{latitude_trend[\'p_value\']:.3f}",\n    "LongitudeSlope": f"{longitude_trend[\'slope_per_decade\']:.2f}",\n    "LongitudeLow": f"{longitude_trend[\'ci_low\']:.2f}",\n    "LongitudeHigh": f"{longitude_trend[\'ci_high\']:.2f}",\n    "LongitudeP": f"{longitude_trend[\'p_value\']:.3f}",\n    "EnergyP": f"{location_energy[\'permutation_p_value\']:.3f}",\n    "DistanceMeanSlope": f"{proximity_all_mean[\'slope_per_decade\']:.1f}",\n    "DistanceMeanLow": f"{proximity_all_mean[\'ci_low\']:.1f}",\n    "DistanceMeanHigh": f"{proximity_all_mean[\'ci_high\']:.1f}",\n    "DistanceMeanP": f"{proximity_all_mean[\'p_value\']:.3f}",\n    "DistanceMedianSlope": (\n        f"{proximity_all_median[\'median_slope_km_per_decade\']:.1f}"\n    ),\n    "DistanceMedianLow": f"{proximity_all_median[\'ci_low\']:.1f}",\n    "DistanceMedianHigh": f"{proximity_all_median[\'ci_high\']:.1f}",\n    "DistanceMedianP": f"{proximity_all_median[\'p_value\']:.3f}",\n    "RelevantDistanceMeanSlope": (\n        f"{proximity_relevant_mean[\'slope_per_decade\']:.1f}"\n    ),\n    "RelevantDistanceMeanLow": f"{proximity_relevant_mean[\'ci_low\']:.1f}",\n    "RelevantDistanceMeanHigh": f"{proximity_relevant_mean[\'ci_high\']:.1f}",\n    "RelevantDistanceMeanP": f"{proximity_relevant_mean[\'p_value\']:.3f}",\n    "EarlyDistanceMedian": (\n        f"{early_recurvers[\'minimum_distance_to_newfoundland_km\'].median():.0f}"\n    ),\n    "LateDistanceMedian": (\n        f"{late_recurvers[\'minimum_distance_to_newfoundland_km\'].median():.0f}"\n    ),\n    "EndpointMinimumN": endpoint_summary["endpoint_count"],\n    "EndpointMinimumPct": (\n        f"{100.0 * endpoint_summary[\'endpoint_fraction\']:.1f}"\n    ),\n    "EarlyEndpointPct": (\n        f"{100.0 * endpoint_summary[\'early_endpoint_fraction\']:.1f}"\n    ),\n    "LateEndpointPct": (\n        f"{100.0 * endpoint_summary[\'late_endpoint_fraction\']:.1f}"\n    ),\n    "NonEndpointN": len(non_endpoint_recurvers),\n    "NonEndpointMeanSlope": (\n        f"{proximity_non_endpoint_mean[\'slope_per_decade\']:.1f}"\n    ),\n    "NonEndpointMeanLow": f"{proximity_non_endpoint_mean[\'ci_low\']:.1f}",\n    "NonEndpointMeanHigh": f"{proximity_non_endpoint_mean[\'ci_high\']:.1f}",\n    "NonEndpointMeanP": f"{proximity_non_endpoint_mean[\'p_value\']:.3f}",\n    "NonEndpointMedianSlope": (\n        f"{proximity_non_endpoint_median[\'median_slope_km_per_decade\']:.1f}"\n    ),\n    "NonEndpointMedianLow": (\n        f"{proximity_non_endpoint_median[\'ci_low\']:.1f}"\n    ),\n    "NonEndpointMedianHigh": (\n        f"{proximity_non_endpoint_median[\'ci_high\']:.1f}"\n    ),\n    "NonEndpointMedianP": (\n        f"{proximity_non_endpoint_median[\'p_value\']:.3f}"\n    ),\n    "ProjectionAlternativeN": projection_summary[\n        "alternative_relevant_events"\n    ],\n    "ProjectionClassificationChanges": projection_summary[\n        "classification_changes"\n    ],\n    "ProjectionDistanceCorrelation": (\n        f"{projection_summary[\'distance_correlation\']:.4f}"\n    ),\n    "ProjectionMedianAbsoluteDifference": (\n        f"{projection_summary[\'median_absolute_difference_km\']:.1f}"\n    ),\n    "ProjectionAlternativeMeanSlope": (\n        f"{alternative_projection_mean_trend[\'slope_per_decade\']:.1f}"\n    ),\n    "ProjectionAlternativeMeanLow": (\n        f"{alternative_projection_mean_trend[\'ci_low\']:.1f}"\n    ),\n    "ProjectionAlternativeMeanHigh": (\n        f"{alternative_projection_mean_trend[\'ci_high\']:.1f}"\n    ),\n    "ProjectionAlternativeMeanP": (\n        f"{alternative_projection_mean_trend[\'p_value\']:.3f}"\n    ),\n    "ThresholdLowN": int(threshold_300["events"]),\n    "ThresholdHighN": int(threshold_1000["events"]),\n    "ThresholdFullIRRMin": (\n        f"{threshold_sensitivity[\'irr_per_decade\'].min():.3f}"\n    ),\n    "ThresholdFullIRRMax": (\n        f"{threshold_sensitivity[\'irr_per_decade\'].max():.3f}"\n    ),\n    "ThresholdSatelliteIRRMin": (\n        f"{threshold_sensitivity[\'satellite_irr_per_decade\'].min():.3f}"\n    ),\n    "ThresholdSatelliteIRRMax": (\n        f"{threshold_sensitivity[\'satellite_irr_per_decade\'].max():.3f}"\n    ),\n    "ThresholdHighSatelliteN": int(threshold_1000["satellite_events"]),\n    "ThresholdHighSatelliteIRR": (\n        f"{threshold_1000[\'satellite_irr_per_decade\']:.3f}"\n    ),\n    "ThresholdHighSatelliteLow": f"{threshold_1000[\'satellite_ci_low\']:.3f}",\n    "ThresholdHighSatelliteHigh": (\n        f"{threshold_1000[\'satellite_ci_high\']:.3f}"\n    ),\n    "ThresholdHighSatelliteP": (\n        f"{threshold_1000[\'satellite_p_value\']:.3f}"\n    ),\n    "DetectorEventMin": int(detector_sensitivity["events"].min()),\n    "DetectorEventMax": int(detector_sensitivity["events"].max()),\n    "DetectorFullIRRMin": (\n        f"{detector_sensitivity[\'irr_per_decade\'].min():.3f}"\n    ),\n    "DetectorFullIRRMax": (\n        f"{detector_sensitivity[\'irr_per_decade\'].max():.3f}"\n    ),\n    "DetectorSatelliteIRRMin": (\n        f"{detector_sensitivity[\'satellite_irr_per_decade\'].min():.3f}"\n    ),\n    "DetectorSatelliteIRRMax": (\n        f"{detector_sensitivity[\'satellite_irr_per_decade\'].max():.3f}"\n    ),\n    "DetectorSatelliteModelSignificantN": int(\n        (detector_sensitivity["satellite_p_value"] < 0.05).sum()\n    ),\n    "DetectorSatelliteHACSignificantN": int(\n        (detector_sensitivity["satellite_hac_p_value"] < 0.05).sum()\n    ),\n    "DetectorSensitiveLatitude": (\n        f"{most_detectable_satellite_detector[\'latitude_gate_deg_n\']:.0f}"\n    ),\n    "DetectorSensitiveWindow": int(\n        most_detectable_satellite_detector["window_hours"]\n    ),\n    "DetectorSensitiveThreshold": (\n        f"{most_detectable_satellite_detector[\'post_east_min_kmh\']:.1f}"\n    ),\n    "DetectorSensitiveSatelliteN": int(\n        most_detectable_satellite_detector["satellite_events"]\n    ),\n    "DetectorSensitiveSatelliteIRR": (\n        f"{most_detectable_satellite_detector[\'satellite_irr_per_decade\']:.3f}"\n    ),\n    "DetectorSensitiveSatelliteLow": (\n        f"{most_detectable_satellite_detector[\'satellite_ci_low\']:.3f}"\n    ),\n    "DetectorSensitiveSatelliteHigh": (\n        f"{most_detectable_satellite_detector[\'satellite_ci_high\']:.3f}"\n    ),\n    "DetectorSensitiveSatelliteP": (\n        f"{most_detectable_satellite_detector[\'satellite_p_value\']:.3f}"\n    ),\n    "TropicalAtTurnN": ts_events,\n}\nmacro_lines = [\n    f"\\\\newcommand{{\\\\{name}}}{{{value}}}" for name, value in macro_values.items()\n]\n(OUTPUT_DIR / "results_macros.tex").write_text("\\n".join(macro_lines) + "\\n")\n\nprint("Analysis complete.")\nprint(\n    f"Baseline N={len(velocity_events)}; "\n    f"IRR={full_count_summary[\'irr_per_decade\']:.3f} "\n    f"({full_count_summary[\'ci_low\']:.3f}-"\n    f"{full_count_summary[\'ci_high\']:.3f})"\n)\n'
path = PROJECT_ROOT / 'run_analysis.py'
path.write_text(source)
print(f"Wrote {path.name} ({len(source):,} characters)")

In [ ]:
from pathlib import Path

source = '"""Rebuild the illustrative satellite-era 500-hPa composite."""\n\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport json\nimport os\nfrom pathlib import Path\nimport time\nfrom urllib.parse import urlencode\nfrom urllib.request import urlopen\n\nimport cartopy\nimport cartopy.crs as ccrs\nimport cartopy.feature as cfeature\nfrom cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\nfrom scipy.interpolate import RegularGridInterpolator\nimport xarray as xr\n\nfrom analysis_core import (\n    SATELLITE_START,\n    YEAR_MAX,\n    load_newfoundland_island_polygon,\n    write_json,\n)\n\n\nROOT = Path(os.environ.get("NL_RECURV_WORKDIR", Path.cwd())).resolve()\nOUTPUT_DIR = ROOT / "outputs"\nFIGURE_DIR = ROOT / "figures"\nCACHE_DIR = Path(\n    os.environ.get("NCEP_EVENT_CACHE", ROOT / "data" / "ncep_event_fields")\n).resolve()\nCARTOPY_DIR = Path(\n    os.environ.get("CARTOPY_DATA_DIR", ROOT / "data" / "cartopy")\n).resolve()\nCLIMATOLOGY_PATH = Path(\n    os.environ.get(\n        "NCEP_HGT_CLIMATOLOGY",\n        ROOT / "data" / "hgt.mon.ltm.1991-2020.nc",\n    )\n).resolve()\n\nEVENTS_PATH = OUTPUT_DIR / "corrected_newfoundland_relevant_events.csv"\nCLIMATOLOGY_URL = (\n    "https://downloads.psl.noaa.gov/Datasets/"\n    "ncep.reanalysis.derived/pressure/hgt.mon.ltm.1991-2020.nc"\n)\nNCSS_TEMPLATE = (\n    "https://psl.noaa.gov/thredds/ncss/grid/Datasets/"\n    "ncep.reanalysis/pressure/hgt.{year}.nc"\n)\n\nLAT_NORTH = 80.0\nLAT_SOUTH = 5.0\nLON_WEST = 220.0\nLON_EAST = 360.0\nLEVEL_HPA = 500.0\n\nBLUE = "#1769AA"\nRED = "#B42318"\n\n\ndef download_file(url: str, destination: Path, attempts: int = 3) -> Path:\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists() and destination.stat().st_size > 1000:\n        return destination\n    temporary = destination.with_suffix(destination.suffix + ".part")\n    for attempt in range(1, attempts + 1):\n        try:\n            with urlopen(url, timeout=90) as response, temporary.open("wb") as stream:\n                while True:\n                    block = response.read(1024 * 1024)\n                    if not block:\n                        break\n                    stream.write(block)\n            temporary.replace(destination)\n            return destination\n        except Exception:\n            if temporary.exists():\n                temporary.unlink()\n            if attempt == attempts:\n                raise\n            time.sleep(attempt * 2)\n    raise RuntimeError("unreachable")\n\n\ndef event_url(row: pd.Series) -> str:\n    event_time = pd.Timestamp(row["recurv_time"])\n    query = urlencode(\n        {\n            "var": "hgt",\n            "north": LAT_NORTH,\n            "west": LON_WEST,\n            "east": LON_EAST,\n            "south": LAT_SOUTH,\n            "horizStride": 1,\n            "time": event_time.strftime("%Y-%m-%dT%H:%M:%SZ"),\n            "vertCoord": LEVEL_HPA,\n            "accept": "netcdf4",\n        }\n    )\n    return f"{NCSS_TEMPLATE.format(year=event_time.year)}?{query}"\n\n\ndef fetch_event(row: pd.Series) -> Path:\n    destination = CACHE_DIR / f"{row[\'sid\']}.nc"\n    return download_file(event_url(row), destination)\n\n\ndef false_discovery_rate_mask(p_values: np.ndarray, q: float = 0.05) -> np.ndarray:\n    flat = np.asarray(p_values, dtype=float).ravel()\n    valid = np.isfinite(flat)\n    ordered = np.argsort(flat[valid])\n    sorted_p = flat[valid][ordered]\n    thresholds = q * np.arange(1, len(sorted_p) + 1) / len(sorted_p)\n    passing = np.flatnonzero(sorted_p <= thresholds)\n    mask = np.zeros_like(flat, dtype=bool)\n    if len(passing):\n        cutoff = sorted_p[passing[-1]]\n        mask = np.isfinite(flat) & (flat <= cutoff)\n    return mask.reshape(p_values.shape)\n\n\ndef add_map_ticks(ax, extent):\n    ax.set_extent(extent, crs=ccrs.PlateCarree())\n    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#F1EFE8")\n    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="#EAF2F8")\n    ax.coastlines("50m", linewidth=0.5)\n    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.3)\n    ax.set_xticks(np.arange(-120, 1, 20), crs=ccrs.PlateCarree())\n    ax.set_yticks(np.arange(20, 81, 10), crs=ccrs.PlateCarree())\n    ax.xaxis.set_major_formatter(LongitudeFormatter())\n    ax.yaxis.set_major_formatter(LatitudeFormatter())\n    ax.grid(alpha=0.25, linestyle="--", linewidth=0.4)\n\n\nif not EVENTS_PATH.exists():\n    raise FileNotFoundError(\n        "Run run_analysis.py before rebuilding the composite"\n    )\n\nCACHE_DIR.mkdir(parents=True, exist_ok=True)\nFIGURE_DIR.mkdir(parents=True, exist_ok=True)\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nCARTOPY_DIR.mkdir(parents=True, exist_ok=True)\ncartopy.config["data_dir"] = str(CARTOPY_DIR)\n\nevents = pd.read_csv(EVENTS_PATH, parse_dates=["recurv_time"])\nevents = events[events["season"] >= SATELLITE_START].copy().reset_index(drop=True)\nprint(f"Baseline satellite-era events: {len(events)}")\n\nif not CLIMATOLOGY_PATH.exists():\n    print("Downloading 1991-2020 monthly climatology")\n    download_file(CLIMATOLOGY_URL, CLIMATOLOGY_PATH)\n\nprint("Downloading/caching event-time fields")\nerrors = []\nwith ThreadPoolExecutor(max_workers=8) as executor:\n    futures = {\n        executor.submit(fetch_event, row): row["sid"]\n        for _, row in events.iterrows()\n    }\n    for completed, future in enumerate(as_completed(futures), start=1):\n        sid = futures[future]\n        try:\n            future.result()\n        except Exception as exc:\n            errors.append({"sid": sid, "error": str(exc)})\n        if completed % 10 == 0 or completed == len(futures):\n            print(f"  {completed}/{len(futures)} fields resolved")\n\nif errors:\n    raise RuntimeError(f"Failed event downloads: {errors}")\n\nclimatology = xr.open_dataset(CLIMATOLOGY_PATH, decode_times=False)["hgt"].sel(\n    level=LEVEL_HPA,\n    lat=slice(LAT_NORTH, LAT_SOUTH),\n    lon=slice(LON_WEST, 357.5),\n)\n\nanomalies = []\nabsolute_fields = []\nevent_metadata = []\nfor _, row in events.iterrows():\n    event_path = CACHE_DIR / f"{row[\'sid\']}.nc"\n    event_ds = xr.open_dataset(event_path)\n    field = (\n        event_ds["hgt"]\n        .sel(level=LEVEL_HPA)\n        .squeeze(drop=True)\n        .sel(lon=slice(LON_WEST, 357.5))\n        .load()\n    )\n    month_index = int(pd.Timestamp(row["recurv_time"]).month - 1)\n    monthly_normal = climatology.isel(time=month_index).load()\n    monthly_normal = monthly_normal.sel(lat=field["lat"], lon=field["lon"])\n    anomaly = field - monthly_normal\n    anomalies.append(np.asarray(anomaly.values, dtype=float))\n    absolute_fields.append(np.asarray(field.values, dtype=float))\n    event_metadata.append(\n        {\n            "sid": row["sid"],\n            "event_time": str(pd.Timestamp(row["recurv_time"])),\n            "recurv_lat": float(row["recurv_lat"]),\n            "recurv_lon": float(row["recurv_lon"]),\n            "source_file": str(event_path.name),\n        }\n    )\n    event_ds.close()\n\nanomaly_stack = np.stack(anomalies)\nabsolute_stack = np.stack(absolute_fields)\nlatitude = np.asarray(field["lat"].values, dtype=float)\nlongitude_360 = np.asarray(field["lon"].values, dtype=float)\nlongitude = np.where(longitude_360 > 180.0, longitude_360 - 360.0, longitude_360)\n\ngeographic_mean = np.nanmean(anomaly_stack, axis=0)\nabsolute_mean = np.nanmean(absolute_stack, axis=0)\ntest = stats.ttest_1samp(anomaly_stack, popmean=0.0, axis=0, nan_policy="omit")\ngeographic_fdr = false_discovery_rate_mask(test.pvalue, q=0.05)\n\n\nrelative_lon = np.arange(-30.0, 30.1, 2.5)\nrelative_lat = np.arange(-20.0, 20.1, 2.5)\nrelative_mesh_lon, relative_mesh_lat = np.meshgrid(relative_lon, relative_lat)\nrelative_anomalies = []\nrelative_absolute = []\n\nascending_lat = latitude[::-1]\nfor event_index, row in events.iterrows():\n    event_lon = float(row["recurv_lon"]) % 360.0\n    event_lat = float(row["recurv_lat"])\n    query = np.column_stack(\n        [\n            (event_lat + relative_mesh_lat).ravel(),\n            (event_lon + relative_mesh_lon).ravel(),\n        ]\n    )\n    anomaly_interpolator = RegularGridInterpolator(\n        (ascending_lat, longitude_360),\n        anomaly_stack[event_index][::-1, :],\n        bounds_error=False,\n        fill_value=np.nan,\n    )\n    absolute_interpolator = RegularGridInterpolator(\n        (ascending_lat, longitude_360),\n        absolute_stack[event_index][::-1, :],\n        bounds_error=False,\n        fill_value=np.nan,\n    )\n    relative_anomalies.append(\n        anomaly_interpolator(query).reshape(relative_mesh_lat.shape)\n    )\n    relative_absolute.append(\n        absolute_interpolator(query).reshape(relative_mesh_lat.shape)\n    )\n\nrelative_anomaly_stack = np.stack(relative_anomalies)\nrelative_absolute_stack = np.stack(relative_absolute)\nrelative_mean = np.nanmean(relative_anomaly_stack, axis=0)\nrelative_absolute_mean = np.nanmean(relative_absolute_stack, axis=0)\nrelative_test = stats.ttest_1samp(\n    relative_anomaly_stack, popmean=0.0, axis=0, nan_policy="omit"\n)\nrelative_fdr = false_discovery_rate_mask(relative_test.pvalue, q=0.05)\n\n\nlevels = np.arange(-60, 61, 10)\nabsolute_levels = np.arange(5400, 5941, 60)\nfig = plt.figure(figsize=(11.2, 5.6))\n\nax1 = fig.add_subplot(121, projection=ccrs.PlateCarree())\nadd_map_ticks(ax1, (-110, 0, 15, 75))\nfilled = ax1.contourf(\n    longitude,\n    latitude,\n    geographic_mean,\n    levels=levels,\n    cmap="RdBu_r",\n    extend="both",\n    transform=ccrs.PlateCarree(),\n)\nheight_contours = ax1.contour(\n    longitude,\n    latitude,\n    absolute_mean,\n    levels=absolute_levels,\n    colors="#333333",\n    linewidths=0.55,\n    transform=ccrs.PlateCarree(),\n)\nax1.clabel(height_contours, fmt="%d", fontsize=6, inline=True)\nstipple_y, stipple_x = np.where(geographic_fdr)\nkeep = (stipple_y % 2 == 0) & (stipple_x % 2 == 0)\nax1.scatter(\n    longitude[stipple_x[keep]],\n    latitude[stipple_y[keep]],\n    s=2.5,\n    color="#222222",\n    alpha=0.65,\n    transform=ccrs.PlateCarree(),\n)\nisland_lonlat, _ = load_newfoundland_island_polygon(CARTOPY_DIR)\nax1.add_geometries(\n    [island_lonlat],\n    crs=ccrs.PlateCarree(),\n    facecolor="none",\n    edgecolor=RED,\n    linewidth=1.2,\n)\nax1.set_title("(a) Geographic composite")\n\nax2 = fig.add_subplot(122)\nax2.contourf(\n    relative_lon,\n    relative_lat,\n    relative_mean,\n    levels=levels,\n    cmap="RdBu_r",\n    extend="both",\n)\nrelative_contours = ax2.contour(\n    relative_lon,\n    relative_lat,\n    relative_absolute_mean,\n    levels=absolute_levels,\n    colors="#333333",\n    linewidths=0.55,\n)\nax2.clabel(relative_contours, fmt="%d", fontsize=6, inline=True)\nstipple_y, stipple_x = np.where(relative_fdr)\nkeep = (stipple_y % 2 == 0) & (stipple_x % 2 == 0)\nax2.scatter(\n    relative_lon[stipple_x[keep]],\n    relative_lat[stipple_y[keep]],\n    s=3,\n    color="#222222",\n    alpha=0.65,\n)\nax2.scatter(0, 0, marker="*", s=95, color="#111111", edgecolor="white", linewidth=0.5)\nax2.axhline(0, color="#777777", linewidth=0.4)\nax2.axvline(0, color="#777777", linewidth=0.4)\nax2.set_xlabel("Longitude offset from recurvature point (degrees)")\nax2.set_ylabel("Latitude offset from recurvature point (degrees)")\nax2.set_title("(b) Storm-relative composite")\nax2.grid(alpha=0.2, linestyle="--", linewidth=0.4)\n\ncolorbar = fig.colorbar(\n    filled,\n    ax=[ax1, ax2],\n    orientation="horizontal",\n    fraction=0.045,\n    pad=0.18,\n    aspect=40,\n)\ncolorbar.set_label("500-hPa geopotential-height anomaly (m)")\nfig.suptitle(\n    f"500-hPa environment at baseline recurvature times "\n    f"({SATELLITE_START}-{YEAR_MAX}; N={len(events)})",\n    y=0.99,\n)\nfig.subplots_adjust(left=0.06, right=0.98, top=0.88, bottom=0.27, wspace=0.19)\nfigure_path = FIGURE_DIR / "fig9_z500_composite.png"\nfig.savefig(figure_path, dpi=300, bbox_inches="tight", facecolor="white")\nplt.close(fig)\nprint("Saved:", figure_path)\n\n\nfield_dataset = xr.Dataset(\n    {\n        "geographic_anomaly": (("lat", "lon"), geographic_mean),\n        "geographic_absolute_height": (("lat", "lon"), absolute_mean),\n        "geographic_fdr_significant": (\n            ("lat", "lon"),\n            geographic_fdr.astype(np.int8),\n        ),\n        "storm_relative_anomaly": (\n            ("relative_lat", "relative_lon"),\n            relative_mean,\n        ),\n        "storm_relative_absolute_height": (\n            ("relative_lat", "relative_lon"),\n            relative_absolute_mean,\n        ),\n        "storm_relative_fdr_significant": (\n            ("relative_lat", "relative_lon"),\n            relative_fdr.astype(np.int8),\n        ),\n    },\n    coords={\n        "lat": latitude,\n        "lon": longitude,\n        "relative_lat": relative_lat,\n        "relative_lon": relative_lon,\n    },\n    attrs={\n        "event_count": int(len(events)),\n        "event_period": f"{SATELLITE_START}-{YEAR_MAX}",\n        "pressure_level_hpa": LEVEL_HPA,\n        "native_grid_spacing_degrees": 2.5,\n        "event_time_resolution": "6-hourly",\n        "storm_relative_interpolation": "bilinear on the native latitude-longitude grid",\n        "climatology": "1991-2020 monthly NCEP/NCAR Reanalysis 1",\n        "significance": "two-sided one-sample t test with Benjamini-Hochberg FDR q=0.05",\n    },\n)\nfield_dataset.to_netcdf(OUTPUT_DIR / "z500_composite_fields.nc")\n\nmetadata = {\n    "event_count": int(len(events)),\n    "period": f"{SATELLITE_START}-{YEAR_MAX}",\n    "pressure_level_hpa": LEVEL_HPA,\n    "native_grid_spacing_degrees": 2.5,\n    "event_time_resolution": "6-hourly",\n    "storm_relative_interpolation": "bilinear on the native latitude-longitude grid",\n    "source": "NCEP/NCAR Reanalysis 1",\n    "climatology": "1991-2020 monthly long-term mean",\n    "geographic_domain": {\n        "latitude": [LAT_SOUTH, LAT_NORTH],\n        "longitude_east": [LON_WEST, LON_EAST],\n    },\n    "storm_relative_domain_degrees": {\n        "latitude_offset": [float(relative_lat.min()), float(relative_lat.max())],\n        "longitude_offset": [float(relative_lon.min()), float(relative_lon.max())],\n    },\n    "significance": (\n        "Two-sided one-sample t test of event anomalies against zero; "\n        "Benjamini-Hochberg false-discovery-rate control at q=0.05."\n    ),\n    "events": event_metadata,\n}\nwrite_json(OUTPUT_DIR / "z500_composite_metadata.json", metadata)\nprint("Composite rebuild complete")\n'
path = PROJECT_ROOT / 'rebuild_composite.py'
path.write_text(source)
print(f"Wrote {path.name} ({len(source):,} characters)")

## 3. Run the track analysis

In [ ]:
import runpy

runpy.run_path(str(PROJECT_ROOT / "run_analysis.py"), run_name="__main__")

## 4. Rebuild the 500-hPa composite

Set `RUN_COMPOSITE = False` when only the track climatology is needed. With the
default `True`, this step downloads and caches the event-time NCEP fields and
recreates the composite.

In [ ]:
RUN_COMPOSITE = True

if RUN_COMPOSITE:
    runpy.run_path(
        str(PROJECT_ROOT / "rebuild_composite.py"),
        run_name="__main__",
    )
else:
    print("Composite skipped")

## 5. Inspect the principal estimates

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = json.loads(
    (PROJECT_ROOT / "outputs" / "analysis_summary.json").read_text()
)

display(pd.DataFrame(
    [
        {
            "period": "1950–2023",
            **summary["frequency"]["full_period"],
        },
        {
            "period": "1979–2023",
            **summary["frequency"]["satellite_era"],
        },
    ]
)[
    [
        "period",
        "total_events",
        "irr_per_decade",
        "ci_low",
        "ci_high",
        "p_value",
    ]
])

print(
    "Baseline relevant events:",
    summary["sample"]["corrected_relevant_events"],
)
print(
    "Tropical-origin source storms:",
    summary["sample"]["tropical_origin_storms"],
)
print(
    "Eligible storms:",
    summary["sample"]["eligible_storms"],
)
print(
    "All baseline recurvers:",
    summary["sample"]["all_corrected_recurvers"],
)

In [ ]:
from IPython.display import Image, display

for filename in [
    "fig3_annual_counts_poisson.png",
    "fig5_recurvature_locations.png",
    "fig6_proximity_all_recurvers.png",
    "fig8_threshold_sensitivity.png",
    "fig9_z500_composite.png",
]:
    path = PROJECT_ROOT / "figures" / filename
    if path.exists():
        display(Image(filename=str(path), width=900))

## 6. Package generated outputs

In [ ]:
import shutil

archive = shutil.make_archive(
    str(PROJECT_ROOT / "newfoundland_recurvature_outputs"),
    "zip",
    root_dir=PROJECT_ROOT,
    base_dir="outputs",
)
print("Created:", archive)
print("Figures remain in:", PROJECT_ROOT / "figures")